# 03 — Preprocessing & Exploratory Data Analysis

## ANTARBODH — AI-Powered Subsurface Ocean Intelligence

This notebook performs exploratory data analysis, quality control,
preprocessing, spatial/temporal harmonization, and preparation of
model-ready input and target datasets.

### Workflow

1. Load audited raw datasets
2. Statistical EDA
3. Distribution analysis
4. Outlier and extreme-value analysis
5. Spatial EDA
6. Missing-value analysis
7. SSS ascending/descending analysis
8. GLORYS depth-wise analysis
9. Multi-variable relationship analysis
10. Temporal EDA
11. Coordinate and time normalization
12. Unit conversion and invalid-value handling
13. Quality control and masking
14. Missing-data treatment
15. Regridding and interpolation
16. Dataset alignment
17. Model-ready X/Y construction
18. Normalization
19. Final QC and leakage checks
20. Save processed data and reports

In [50]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import dask

In [ ]:

# Project root
PROJECT_ROOT = Path("..")

# Raw data
RAW_DIR = PROJECT_ROOT / "data" / "raw"

GLORYS_DIR = RAW_DIR / "glorys"
SST_DIR = RAW_DIR / "sst"
SSS_DIR = RAW_DIR / "sss"
SSH_DIR = RAW_DIR / "ssh"
CURRENTS_DIR = RAW_DIR / "currents"
WINDS_DIR = RAW_DIR / "winds"
ARGO_DIR = RAW_DIR / "argo"

# Output directories
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
REPORTS_DIR = OUTPUT_DIR / "reports"
STATS_DIR = OUTPUT_DIR / "statistics"

# Create output directories if they do not exist
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data:", RAW_DIR.resolve())
print("Outputs:", OUTPUT_DIR.resolve())

In [ ]:
glorys = xr.open_dataset(
    GLORYS_DIR / "glorys_bob_(2).nc",
    chunks={
        "time": 7,
        "depth": -1,
        "latitude": 60,
        "longitude": 80,
    }
)

sst = xr.open_dataset(
    SST_DIR / "sst_bob_(2).nc",
    chunks={"time": 7}
)

sss_asc = xr.open_dataset(
    SSS_DIR / "sss_bob_ascending_(2).nc",
    chunks={"time": 7}
)

sss_desc = xr.open_dataset(
    SSS_DIR / "sss_bob_descending_(2).nc",
    chunks={"time": 7}
)

ssh = xr.open_dataset(
    SSH_DIR / "ssh_bob_(2).nc",
    chunks={"time": 7}
)

currents = xr.open_dataset(
    CURRENTS_DIR / "currents_bob_(2).nc",
    chunks={"time": 7}
)

winds = xr.open_dataset(
    WINDS_DIR / "winds_bob_(2).nc",
    chunks={"time": 7}
)

In [ ]:
datasets = {
    "GLORYS": glorys,
    "SST": sst,
    "SSS Ascending": sss_asc,
    "SSS Descending": sss_desc,
    "SSH": ssh,
    "Currents": currents,
    "Winds": winds,
}

for name, ds in datasets.items():
    print(f"{name}:")
    print(ds)
    print("-" * 80)

In [ ]:
for name, ds in datasets.items():
    print(f"{name}: {list(ds.data_vars)}")
    print(ds.chunks)

In [ ]:
for name, ds in datasets.items():
    if "time" in ds.coords:
        print(
            f"{name}:",
            ds["time"].values
        )
        print(ds.time.min().compute())
        print(ds.time.max().compute())

In [ ]:
for name, ds in datasets.items():
    print(f"{name}: {dict(ds.sizes)}")

## 2. Statistical EDA

The objective of this section is to characterize the statistical
properties of the raw variables before applying any preprocessing.

We examine:
- central tendency
- spread
- quantiles
- minimum and maximum values
- missing values
- potential extreme values

No values are modified during this stage.

In [ ]:
# Variables used in the ANTARBODH prototype

eda_variables = {
    "SST": sst["sea_surface_temperature"],
    "SSS Ascending": sss_asc["Sea_Surface_Salinity"],
    "SSS Descending": sss_desc["Sea_Surface_Salinity"],
    "SSH": ssh["sla"],
    "Current U": currents["uo"],
    "Current V": currents["vo"],
    "Wind U": winds["eastward_wind"],
    "Wind V": winds["northward_wind"],
    "GLORYS Temperature": glorys["thetao"],
}

print(f"Number of variables: {len(eda_variables)}")

for name, da in eda_variables.items():
    print(f"{name:25s} -> {da.dims}")

In [ ]:
# Descriptive statistics for each variable
# 5-year Dask-friendly audit version

stats = {}

for name, da in eda_variables.items():

    print(f"\nProcessing: {name}")
    print("-" * 40)

    # Keep only finite values
    valid = da.where(np.isfinite(da))

    # Full-data statistics
    missing_percent = (
        100 * valid.isnull().mean()
    ).compute().item()

    mean_value = (
        valid.mean(skipna=True)
    ).compute().item()

    std_value = (
        valid.std(skipna=True)
    ).compute().item()

    stats[name] = {
        "missing_percent": missing_percent,
        "mean": mean_value,
        "std": std_value,
    }

    print(f"Missing %: {missing_percent:.2f}")
    print(f"Mean:      {mean_value:.4f}")
    print(f"Std:       {std_value:.4f}")

print("\n========== DESCRIPTIVE STATISTICS ==========")
stats

In [59]:
sample_dates = [
    "2020-01-15",
    "2020-04-15",
    "2020-07-15",
    "2020-10-15",
    "2021-01-15",
    "2022-01-15",
    "2023-01-15",
    "2024-01-15",
    "2025-01-15",
]

In [ ]:
sample_stats = {}

for name, da in eda_variables.items():

    print(f"\nProcessing sample: {name}")
    print("-" * 40)

    if "time" in da.dims:
        sample = da.sel(
            time=sample_dates,
            method="nearest"
        )
    else:
        sample = da

    valid = sample.where(np.isfinite(sample))

    sample_stats[name] = {
        "min": float(valid.min(skipna=True).compute()),
        "q25": float(valid.quantile(0.25, skipna=True).compute()),
        "median": float(valid.quantile(0.50, skipna=True).compute()),
        "mean": float(valid.mean(skipna=True).compute()),
        "q75": float(valid.quantile(0.75, skipna=True).compute()),
        "max": float(valid.max(skipna=True).compute()),
        "std": float(valid.std(skipna=True).compute()),
    }

print("\n========== SAMPLE DESCRIPTIVE STATISTICS ==========")
sample_stats

In [ ]:
quantile_levels = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]

quantiles = {}

for name, da in eda_variables.items():

    valid = da.where(np.isfinite(da))

    q = valid.quantile(
        quantile_levels,
        skipna=True
    ).compute()

    quantiles[name] = q

quantiles

In [ ]:
stats_file = STATS_DIR / "raw_statistical_eda_1_month.txt"

with open(stats_file, "w") as f:
    f.write("ANTARBODH — Raw Statistical EDA\n")
    f.write("=" * 50 + "\n\n")

    for name, values in stats.items():
        f.write(f"{name}\n")
        f.write("-" * len(name) + "\n")

        for statistic, value in values.items():
            f.write(f"{statistic}: {value}\n")

        f.write("\n")

print(f"Saved statistics to: {stats_file}")

In [ ]:
print("SST units:", sst["sea_surface_temperature"].attrs.get("units"))
print("SST standard name:", sst["sea_surface_temperature"].attrs.get("standard_name"))

### Distribution Analysis

In [ ]:
import random
import pandas as pd

# Generate 30 random dates until 2025
random_dates = pd.date_range(start="2020-01-01", end="2025-12-31", freq="D")
random_dates = random.sample(list(random_dates), 30)
sample_dates = []

# Convert to string format and add to the sample_dates list
sample_dates.extend(date.strftime("%Y-%m-%d") for date in random_dates)

# Sort the dates for better readability
sample_dates = sorted(sample_dates)

print(sample_dates)

In [ ]:
##SST

sst_sample = sst["sea_surface_temperature"].sel(
    time=sample_dates,
    method="nearest"
)


sst_values= sst_sample.values
sst_values=sst_values[np.isfinite(sst_values)] #filtering out invalid values

plt.figure(figsize=(10,6))

plt.hist(sst_values, bins=50, color='skyblue', edgecolor='black')

plt.xlabel("Sea Surface Temperature (°C)")
plt.ylabel("Frequency")
plt.title("Histogram of Sea Surface Temperature (SST)")

plt.tight_layout()

plt.savefig(FIGURES_DIR / "sst_distribution_raw.png",dpi=150,bbox_inches='tight')  

plt.show()

SST
* No obvious pathological values are visible.
* The distribution is not symmetric; it has a noticeable cooler tail.
* The raw values are in Kelvin, so the later preprocessing step will convert them to °C.
* The distribution itself does not justify removing extreme values.
* The major issue with SST remains its high missingness, which we need to investigate spatially.

Conclusion: 🟢 Values look reasonable; investigate missing-data pattern.

In [ ]:
sss_asc_sample = sss_asc["Sea_Surface_Salinity"].sel(
    time=sample_dates,
    method="nearest"
)

sss_desc_sample = sss_desc["Sea_Surface_Salinity"].sel(
    time=sample_dates,
    method="nearest"
)

sss_asc_values = sss_asc_sample.values
sss_asc_values = sss_asc_values[np.isfinite(sss_asc_values)]

sss_desc_values = sss_desc_sample.values
sss_desc_values = sss_desc_values[np.isfinite(sss_desc_values)]

plt.figure(figsize=(8, 5))

plt.hist(sss_asc_values, bins=50, alpha=0.6, label="Ascending")
plt.hist(sss_desc_values, bins=50, alpha=0.6, label="Descending")

plt.xlabel("Sea Surface Salinity")
plt.ylabel("Frequency")
plt.title("SSS Distribution — Raw Data")
plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "sss_distribution_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

Inference:

* Ascending and descending SSS do not have identical statistical behavior.
* The ascending product contains potentially suspicious high-salinity observations.
* However, the graph alone does not prove that those values are erroneous.
* We should investigate where those high values occur geographically and whether they occur in isolated pixels or coherent regions.
* The two datasets should therefore not simply be blindly averaged yet.

Conclusion: 🔴 Requires dedicated SSS QC/outlier investigation.

In [ ]:
ssh_sample = ssh["sla"].sel(
    time=sample_dates,
    method="nearest"
)

ssh_values = ssh_sample.values
ssh_values = ssh_values[np.isfinite(ssh_values)]

plt.figure(figsize=(8, 5))

plt.hist(ssh_values, bins=50)

plt.xlabel("Sea Level Anomaly (m)")
plt.ylabel("Frequency")
plt.title("SSH/SLA Distribution — Raw Data")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "ssh_distribution_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

Inference:

* The distribution is broadly centered around zero.
* Positive and negative anomalies are expected for SLA.
* No obvious pathological spike or impossible-looking distribution is visible.
* The slight positive skew is not itself a problem.

Conclusion: 🟢 No immediate anomaly; retain for further spatial/QC analysis.

In [ ]:
uo_sample = currents["uo"].sel(
    time=sample_dates,
    method="nearest"
)

vo_sample = currents["vo"].sel(
    time=sample_dates,
    method="nearest"
)

uo_values = uo_sample.values
uo_values = uo_values[np.isfinite(uo_values)]

vo_values = uo_sample.values
vo_values = vo_values[np.isfinite(vo_values)]

plt.figure(figsize=(8, 5))

plt.hist(uo_values, bins=50, alpha=0.6, label="U Current")
plt.hist(vo_values, bins=50, alpha=0.6, label="V Current")

plt.xlabel("Current Velocity (m/s)")
plt.ylabel("Frequency")
plt.title("Surface Current Component Distributions")
plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "current_distribution_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

Inference:

* The distributions look continuous rather than containing obvious isolated numerical spikes.
* The tails should nevertheless be checked spatially.
* We should not remove negative values.
* We should eventually consider current speed as an additional diagnostic:
$$ Speed = \sqrt{U^2 + V^2} $$

* but we don't necessarily need to make speed a model input because our planned input channels remain U/V.

Conclusion: 🟢 No obvious anomaly; perform physical/extreme-value QC later.

In [ ]:
wind_u_sample = winds["eastward_wind"].sel(
    time=sample_dates,
    method="nearest"
)

wind_v_sample = winds["northward_wind"].sel(
    time=sample_dates,
    method="nearest"
)


wind_u_values = wind_u_sample.values
wind_u_values = wind_u_values[np.isfinite(wind_u_values)]

wind_v_values = wind_v_sample.values
wind_v_values = wind_v_values[np.isfinite(wind_v_values)]

plt.figure(figsize=(8, 5))

plt.hist(wind_u_values, bins=50, alpha=0.6, label="U Wind")
plt.hist(wind_v_values, bins=50, alpha=0.6, label="V Wind")

plt.xlabel("Wind Velocity (m/s)")
plt.ylabel("Frequency")
plt.title("Surface Wind Component Distributions")
plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "wind_distribution_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

Inference:

* Values themselves don't immediately look like impossible numerical values.
* The strong asymmetry is worth investigating.
* The much bigger concern is the 72.4% missingness.
* The spatial pattern of the available observations is therefore critical.
* We should investigate whether the missing values follow the ASCAT swath pattern.

Conclusion: 🟡 Values require some investigation, but missingness/swath coverage is the bigger issue.

### Distribution EDA — Initial Inference

The raw-variable distributions provide several important observations.

1. SST shows a broad but physically plausible distribution. No obvious
   pathological values are evident, although its high missing-data
   fraction requires spatial investigation.

2. SSS shows substantially different distributions between ascending
   and descending observations. The ascending product has a broader
   distribution and a pronounced high-salinity tail reaching ~40–41.
   These values should not be removed automatically and require
   spatial and overlap-based QC.

3. SSH/SLA is approximately centered around zero with a reasonable
   spread and no obvious pathological distribution.

4. Surface current components show directional asymmetry, particularly
   for the U component. Negative values represent physical direction
   and should not be treated as invalid values. Extreme velocities
   require further spatial/physical investigation.

5. Surface wind components show strong asymmetry, particularly in the
   U component. The values do not immediately indicate invalid data,
   but the high missing-data fraction and observation/swath pattern
   require further investigation.

6. GLORYS temperature should be analyzed separately by depth because
   combining all depths into one distribution masks the vertical
   temperature structure.

Overall, no variable should be cleaned solely on the basis of these
histograms. The next stage is targeted outlier and spatial analysis,
particularly for SSS and the high-missingness surface observations.

## 2B. SSS Ascending/Descending Outlier Investigation

The distribution analysis showed that the ascending SSS product has a
broader distribution and a high-value tail reaching approximately
40–41, while the descending product is more concentrated.

Before applying any QC rule, we investigate:

1. SSS metadata and units
2. Distribution quantiles
3. Number of extreme observations
4. Spatial location of extreme observations
5. Agreement between ascending and descending observations
6. Whether extreme values occur in coherent regions or isolated pixels

No observations are removed during this analysis.

### SSS Metadata Interpretation

Both SSS ascending and descending datasets represent practical
sea surface salinity.

The variable metadata references associated error and QC variables,
but the current downloaded subset contains only the primary
Sea_Surface_Salinity variable.

Therefore, this investigation uses the available salinity field,
metadata-defined validity information, distributional analysis,
spatial structure, and ascending/descending comparison.

The broad metadata limits are not used as direct physical
outlier thresholds.

In [ ]:
sss_asc_da = sss_asc["Sea_Surface_Salinity"]
sss_desc_da = sss_desc["Sea_Surface_Salinity"]

quantile_levels = [0.90, 0.95, 0.975, 0.99, 0.995, 0.999]

asc_quantiles = (
    sss_asc_da
    .where(np.isfinite(sss_asc_da))
    .quantile(quantile_levels)
    .compute()
)

desc_quantiles = (
    sss_desc_da
    .where(np.isfinite(sss_desc_da))
    .quantile(quantile_levels)
    .compute()
)

print("SSS Ascending quantiles")
print("-" * 35)

for q, value in zip(quantile_levels, asc_quantiles.values):
    print(f"{q * 100:5.1f}% : {value:.3f}")

print("\nSSS Descending quantiles")
print("-" * 35)

for q, value in zip(quantile_levels, desc_quantiles.values):
    print(f"{q * 100:5.1f}% : {value:.3f}")

In [ ]:
thresholds = [35, 36, 38, 40]

print("SSS Ascending extreme-value counts")
print("-" * 45)

asc_valid_count = sss_asc_da.count().compute().item()

for threshold in thresholds:
    count = (
        sss_asc_da
        .where(sss_asc_da > threshold)
        .count()
        .compute()
        .item()
    )

    percentage = count / asc_valid_count * 100

    print(
        f"> {threshold:2d}: "
        f"{count:8d} observations "
        f"({percentage:.3f}%)"
    )


print("\nSSS Descending extreme-value counts")
print("-" * 45)

desc_valid_count = sss_desc_da.count().compute().item()

for threshold in thresholds:
    count = (
        sss_desc_da
        .where(sss_desc_da > threshold)
        .count()
        .compute()
        .item()
    )

    percentage = count / desc_valid_count * 100

    print(
        f"> {threshold:2d}: "
        f"{count:8d} observations "
        f"({percentage:.3f}%)"
    )

In [ ]:
asc = sss_asc_da
desc = sss_desc_da

overlap = (
    asc.notnull()
    & desc.notnull()
)

overlap_count = overlap.sum().compute().item()
asc_valid_count = asc.notnull().sum().compute().item()
desc_valid_count = desc.notnull().sum().compute().item()

print("SSS Ascending/Descending overlap")
print("-" * 40)
print("Overlap observations:", overlap_count)
print("Ascending valid observations:", asc_valid_count)
print("Descending valid observations:", desc_valid_count)

In [ ]:
overlap_from_asc = (
    overlap.sum() / asc.notnull().sum()
).compute().item() * 100

overlap_from_desc = (
    overlap.sum() / desc.notnull().sum()
).compute().item() * 100

print(f"Overlap relative to ascending valid data: {overlap_from_asc:.3f}%")
print(f"Overlap relative to descending valid data: {overlap_from_desc:.3f}%")

In [ ]:
high_asc = asc > 35

high_asc_overlap = high_asc & overlap

high_asc_count = high_asc.sum().compute().item()
high_asc_with_desc = high_asc_overlap.sum().compute().item()

print("High ascending SSS observations")
print("-" * 45)
print("Ascending SSS > 35:", high_asc_count)
print(
    "SSS > 35 with descending counterpart:",
    high_asc_with_desc
)

if high_asc_count > 0:
    print(
        "Percentage with descending counterpart:",
        f"{high_asc_with_desc / high_asc_count * 100:.3f}%"
    )

In [26]:
sss_asc_high = (
    sss_asc_da > 35
)

monthly_high_count = (
    sss_asc_high
    .resample(time="1MS")
    .sum()
    .compute()
)

In [ ]:
plt.figure(figsize=(12, 5))

monthly_high_count.plot()

plt.xlabel("Time")
plt.ylabel("Number of observations > 35")
plt.title("Monthly Count of SSS Ascending Observations > 35")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "sss_ascending_extreme_temporal.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
monthly_valid = (
    sss_asc_da
    .notnull()
    .resample(time="1MS")
    .sum()
)

monthly_extreme = (
    (sss_asc_da > 35)
    .resample(time="1MS")
    .sum()
)

monthly_extreme_percentage = (
    monthly_extreme / monthly_valid * 100
).compute()

In [ ]:
plt.figure(figsize=(12, 5))

monthly_extreme_percentage.plot()

plt.xlabel("Time")
plt.ylabel("Percentage of valid observations (%)")
plt.title("Monthly Percentage of SSS Ascending Observations > 35")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "sss_ascending_extreme_percentage_temporal.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [30]:
sss_asc_extreme_frequency = (
    (sss_asc_da > 35)
    .sum(dim="time")
)

sss_asc_valid_frequency = (
    sss_asc_da.notnull()
    .sum(dim="time")
)

sss_asc_extreme_percentage = (
    sss_asc_extreme_frequency
    / sss_asc_valid_frequency
    * 100
)

In [ ]:
sss_asc_extreme_percentage.plot(
    figsize=(9, 6)
)

plt.title(
    "Frequency of SSS Ascending Observations > 35"
)
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "sss_ascending_extreme_frequency.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

### SSS Outlier Investigation — Conclusion

The SSS ascending product exhibits a broader upper-tail distribution
than the descending product.

High-salinity observations are treated as candidate extreme
observations rather than automatically invalid measurements.

Their interpretation is based on:

- distributional quantiles
- frequency of extreme observations
- temporal persistence
- spatial coherence
- ascending/descending overlap where available

The ascending and descending products have limited simultaneous
spatial coverage, so absence of a descending counterpart does not by
itself establish that an ascending observation is invalid.

Therefore, no blanket threshold-based removal of SSS observations is
applied at this stage.

The extreme observations will remain available for downstream QC,
while observation masks preserve the distinction between observed
and missing data.

In [ ]:
print("SSS Ascending variables:")
print(list(sss_asc.variables))

print("\nSSS Descending variables:")
print(list(sss_desc.variables))

In [ ]:
print("\nSSS Ascending data variables:")
print(list(sss_asc.data_vars))

print("\nSSS Descending data variables:")
print(list(sss_desc.data_vars))

## 3. Spatial Exploratory Data Analysis

This section examines the spatial structure of the raw observations.

The objectives are to:

- visualize the spatial distribution of each surface variable
- identify spatially coherent features
- identify land/coastal effects
- inspect extreme-value locations
- understand the spatial pattern of missing observations
- assess whether missing values appear as isolated gaps or large unsupported regions

No interpolation, gap filling, regridding, or value modification is performed
during this stage.

In [34]:
sample_dates = [
    "2020-01-15",
    "2020-04-15",
    "2020-07-15",
    "2020-10-15",
    "2021-01-15",
    "2022-01-15",
    "2023-01-15",
    "2024-01-15",
    "2025-01-15"
    "2021-06-15",
    "2022-06-15",
    "2023-06-15",
    "2024-06-15",
    "2025-06-15"
]

In [ ]:
plt.figure(figsize=(9, 6))

sst["sea_surface_temperature"].isel(time=0).plot(
    cmap="turbo"
)

plt.title("SST — Raw Spatial Distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "sst_spatial_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

What we see

The SST field has a very clear spatial gradient.

Broadly:

warmer temperatures dominate the southern part
cooler temperatures appear toward the north
there are localized structures and gradients
there are also gaps within the ocean coverage

The values are around 297–303 K, consistent with our earlier statistics.

Inference

The SST field is spatially coherent and physically structured.

That's good because SST is likely to be one of the strongest surface predictors for subsurface temperature.

Also notice something important:

The SST spatial structure looks broadly similar to the shallow GLORYS temperature structure.

That doesn't mean they're identical — and they shouldn't be — but the broad thermal patterns are consistent.

In [ ]:
sst_missing = sst["sea_surface_temperature"].isel(time=0).isnull()

plt.figure(figsize=(9, 6))

sst_missing.plot()

plt.title("SST — Missing Data Pattern")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "sst_missing_pattern.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

What we see

The missing regions are not randomly scattered.

There are:

large coherent missing regions
irregular boundaries
smaller holes within observed regions
substantial areas of missing data

This explains our earlier:

58.06% missing

statistic.

Inference

The 58% number looks alarming by itself, but the map gives us the actual story:

SST missingness has strong spatial structure.

Therefore, we shouldn't say:

"SST has too many missing values, so the dataset is unusable."

Instead:

"SST has substantial but spatially structured missingness, requiring conservative gap treatment and an observation mask."

In [ ]:
plt.figure(figsize=(9, 6))

ssh["sla"].isel(time=0).plot(
    cmap="RdBu_r",
    center=0
)

plt.title("SSH/SLA — Raw Spatial Distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "ssh_spatial_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

SSH/SLA — raw spatial distribution

This is a very good-looking oceanographic field.

What we see

* There are alternating positive and negative sea-level anomalies across the Bay of Bengal.

For example:

* positive anomalies around parts of the western/central domain
* negative anomalies toward other regions
* localized strong positive/negative features
* relatively smooth spatial transitions

Inference

* The SSH/SLA field contains mesoscale spatial structure rather than behaving like noise.

* These structures can be associated with ocean circulation features such as eddies and large-scale dynamical variability.

For our project, this is particularly important because SSH/SLA is one of the surface variables that potentially carries information about subsurface thermal structure.

Preprocessing implication

 * Preserve positive and negative values.

 * Keep the original sign.

 * Do not convert negative SLA to zero.

 * Do not use a generic positive-only normalization.

Our normalization later should be something like standardized scaling:

$$ x' = \frac{x-\mu}{\sigma} $$

using training data statistics.

In [ ]:
plt.figure(figsize=(9, 6))

currents["uo"].isel(time=0, depth=0).plot(
    cmap="RdBu_r",
    center=0
)

plt.title("Surface Current U — Raw Spatial Distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "current_u_spatial_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

currents["vo"].isel(time=0, depth=0).plot(
    cmap="RdBu_r",
    center=0
)

plt.title("Surface Current V — Raw Spatial Distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "current_v_spatial_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

winds["eastward_wind"].isel(time=0).plot(
    cmap="RdBu_r",
    center=0
)

plt.title("Surface Wind U — Raw Spatial Distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "wind_u_spatial_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

What we see

Wind U isn't available over the entire domain.

Instead, there are two major observed spatial regions/footprints with substantial gaps between them.

The field itself is spatially structured within those footprints.

Inference

This is characteristic of the sampling/coverage pattern of the selected wind product, rather than evidence that wind velocity is zero outside the observed area.

This is crucial.

In [ ]:
plt.figure(figsize=(9, 6))

winds["northward_wind"].isel(time=0).plot(
    cmap="RdBu_r",
    center=0
)

plt.title("Surface Wind V — Raw Spatial Distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "wind_v_spatial_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

The V component shows a similar observation footprint but a different spatial pattern in the actual values.

What we see
Stronger negative values toward parts of the northern/western observed footprint.
More moderate values elsewhere.
Some positive regions exist.
The eastern/central missing regions correspond largely to lack of observations rather than zero wind.
Inference

The U and V components contain different information, which is exactly what we want.

Together:

$$ \vec{W}=(U,V) $$

describe the surface wind vector.

In [ ]:
plt.figure(figsize=(9, 6))

sss_desc["Sea_Surface_Salinity"].isel(time=0).plot(
    cmap="viridis"
)

plt.title("SSS Descending — Raw Spatial Distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "sss_descending_spatial_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
sss_desc_missing = (
    sss_desc["Sea_Surface_Salinity"]
    .isel(time=0)
    .isnull()
)

plt.figure(figsize=(9, 6))

sss_desc_missing.plot()

plt.title("SSS Descending — Missing Data Pattern")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "sss_descending_missing_pattern.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

What we see

The descending product has much broader coverage than the ascending product on this particular day, but it is still far from complete domain coverage.

There is a large contiguous observed region in the western/central portion of the domain, while much of the eastern side has no observations.

Inference

Again, missingness is structured, not random.

Also, ascending and descending clearly have different spatial sampling patterns.

This explains why their overlap was only about:

0.57%

for this particular sample.

This plot makes the previous point even clearer.

Remember:

yellow = missing
purple = observed

What we see

There is a very large contiguous missing region.

The observed region forms a broad spatial footprint rather than random scattered pixels.

Inference

This is observation-coverage missingness, not simply random sensor noise.

That means filling the entire yellow region using spatial interpolation would be dangerous.

In [ ]:
plt.figure(figsize=(9, 6))

glorys["thetao"].isel(time=0, depth=0).plot(
    cmap="turbo"
)

plt.title(
    f"GLORYS Temperature — Shallowest Available Depth "
    f"({glorys.depth.isel(depth=0).item():.2f} m)"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "glorys_surface_spatial_raw.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

What we see
* Temperature is spatially structured, not random.
* The southern part of the domain is generally warmer, while the northern part is cooler.
* There are clear curved/mesoscale structures in the field.
* Temperatures are approximately 23–31°C in the plotted region.
* There are sharp gradients around some regions, especially toward the northern Bay of Bengal.
* There are also land/coastal/bathymetry-related missing regions.

Inference

This is exactly the kind of structure we expect from an ocean temperature field.

| Variable        | Main spatial observation                        | Preliminary preprocessing implication               |
| --------------- | ----------------------------------------------- | --------------------------------------------------- |
| **SST**         | Strong coherent thermal field + structured gaps | Convert K→°C; conservative interpolation + mask     |
| **SSS Asc**     | Very sparse, structured sampling                | Combine with descending; retain mask                |
| **SSS Desc**    | Broader but still incomplete coverage           | Combine with ascending; retain mask                 |
| **SSH/SLA**     | Strong positive/negative spatial structures     | Preserve sign and spatial variability               |
| **Current U/V** | Coherent velocity structures + missing regions  | Keep U/V separately; preserve masks                 |
| **Wind U/V**    | Strong sampling footprint + spatial variability | Keep U/V separately; never equate missing with zero |
| **GLORYS θ**    | Smooth/coherent thermal structures              | Preserve spatial structure; inspect depth-wise NaNs |


## 6. GLORYS Depth-wise Exploratory Data Analysis

The ANTARBODH target consists of subsurface temperature at 15 target
depths ranging from 0 m to 1000 m.

Before performing vertical interpolation, this section examines:

- temperature statistics as a function of depth
- the vertical temperature structure
- missing-data percentage at each native GLORYS depth
- the availability of data near the deepest target depth
- whether the selected 15 target depths are supported by the native GLORYS profile

No interpolation or modification of the GLORYS target is performed in
this section.

In [ ]:
print("Number of native GLORYS depth levels:", glorys.sizes["depth"])

print("\nNative GLORYS depths:")
print(glorys["depth"].values)

In [ ]:
# ============================================================
# GLORYS — Depth-wise Statistics
# ============================================================

thetao = glorys["thetao"]

depth_dims = [
    "time",
    "latitude",
    "longitude"
]

depth_valid_count = (
    thetao.notnull()
    .sum(dim=depth_dims)
)

depth_missing_fraction = (
    thetao.isnull()
    .mean(dim=depth_dims)
)

depth_min = (
    thetao.min(
        dim=depth_dims,
        skipna=True
    )
)

depth_mean = (
    thetao.mean(
        dim=depth_dims,
        skipna=True
    )
)

depth_max = (
    thetao.max(
        dim=depth_dims,
        skipna=True
    )
)

depth_std = (
    thetao.std(
        dim=depth_dims,
        skipna=True
    )
)

glorys_depth_stats = xr.Dataset(
    {
        "valid_count": depth_valid_count,
        "missing_fraction": depth_missing_fraction,
        "min": depth_min,
        "mean": depth_mean,
        "max": depth_max,
        "std": depth_std,
    }
)

# Execute the complete calculation together
glorys_depth_stats = glorys_depth_stats.compute()

glorys_depth_stats

In [ ]:
# ============================================================
# GLORYS — Mean Temperature Profile
# ============================================================

mean_profile = (
    thetao
    .mean(
        dim=[
            "time",
            "latitude",
            "longitude"
        ],
        skipna=True
    )
    .compute()
)

plt.figure(figsize=(7, 8))

plt.plot(
    mean_profile.values,
    glorys["depth"].values
)

plt.gca().invert_yaxis()

plt.xlabel("Mean Temperature (°C)")
plt.ylabel("Depth (m)")
plt.title(
    "GLORYS Mean Temperature Profile"
)

plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "glorys_mean_temperature_profile.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# GLORYS — Missing Data Percentage by Depth
# ============================================================
thetao = glorys["thetao"]

missing_by_depth = (
    thetao
    .isnull()
    .mean(
        dim=[
            "time",
            "latitude",
            "longitude"
        ]
    )
    * 100
)

missing_by_depth = missing_by_depth.compute()

plt.figure(figsize=(7, 8))

plt.plot(
    missing_by_depth.values,
    glorys["depth"].values
)

plt.gca().invert_yaxis()

plt.xlabel("Missing Data (%)")
plt.ylabel("Depth (m)")
plt.title(
    "GLORYS Missing Data Percentage by Depth"
)

plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "glorys_missingness_by_depth.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# ANTARBODH Target Depths
# ============================================================

target_depths = np.array([
    0,
    5,
    10,
    20,
    30,
    50,
    75,
    100,
    125,
    150,
    200,
    300,
    500,
    700,
    1000
])

print(
    "ANTARBODH target depths (m):"
)

print(target_depths)

In [ ]:
# ============================================================
# GLORYS — Target Depth Support Check
# ============================================================

native_depths = glorys["depth"].values

print(
    "ANTARBODH Target Depth → "
    "Nearest Native GLORYS Level"
)

print("=" * 70)

for target_depth in target_depths:

    nearest_index = np.abs(
        native_depths - target_depth
    ).argmin()

    nearest_depth = float(
        native_depths[nearest_index]
    )

    difference = abs(
        nearest_depth - target_depth
    )

    print(
        f"{target_depth:>5.0f} m → "
        f"{nearest_depth:>9.3f} m "
        f"(difference = {difference:.3f} m)"
    )

In [ ]:
# ============================================================
# GLORYS — 1000 m Vertical Support Check
# ============================================================

target_depth = 1000.0

shallower_depths = native_depths[
    native_depths <= target_depth
]

deeper_depths = native_depths[
    native_depths >= target_depth
]

print(
    "1000 m target support"
)

print("=" * 50)

if (
    len(shallower_depths) > 0
    and len(deeper_depths) > 0
):

    lower = float(
        shallower_depths.max()
    )

    upper = float(
        deeper_depths.min()
    )

    print(
        f"Shallower native level : "
        f"{lower:.3f} m"
    )

    print(
        f"Deeper native level    : "
        f"{upper:.3f} m"
    )

    print(
        "Result: 1000 m is bracketed "
        "and can be vertically interpolated."
    )

else:

    print(
        "Result: 1000 m is outside "
        "the native GLORYS depth range."
    )

In [ ]:
# ============================================================
# GLORYS — 0 m Target Support Check
# ============================================================

native_surface_depth = float(
    native_depths[0]
)

print(
    f"Shallowest native GLORYS level: "
    f"{native_surface_depth:.3f} m"
)

if native_surface_depth > 0:

    print(
        "\n0 m is above the shallowest native level."
    )

    print(
        "Use the shallowest native level as the "
        "0 m proxy rather than blind extrapolation."
    )

else:

    print(
        "\nA native 0 m level is available."
    )

In [ ]:
# ============================================================
# GLORYS — Availability at ANTARBODH Target Depths
# ============================================================

target_availability = []

for target_depth in target_depths:

    field = thetao.sel(
        depth=target_depth,
        method="nearest"
    )

    valid_fraction = (
        field
        .notnull()
        .mean()
        .compute()
        .item()
        * 100
    )

    native_depth = float(
        field["depth"]
        .compute()
        .item()
    )

    target_availability.append(
        {
            "target_depth_m": target_depth,
            "nearest_native_depth_m": native_depth,
            "valid_fraction_percent": valid_fraction,
        }
    )

target_availability

In [ ]:
# ============================================================
# GLORYS — Temperature Near 1000 m
# Representative Date
# ============================================================

glorys_1000m = thetao.sel(
    depth=1000,
    method="nearest"
)

nearest_depth_1000m = float(
    glorys_1000m["depth"]
    .compute()
    .item()
)

field_1000m = glorys_1000m.sel(
    time="2020-01-15",
    method="nearest"
)

plt.figure(figsize=(9, 6))

field_1000m.plot.pcolormesh(
    cmap="turbo"
)

plt.title(
    f"GLORYS Temperature Near 1000 m "
    f"(Native Level: {nearest_depth_1000m:.2f} m)"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "glorys_1000m_temperature.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

### GLORYS Depth-wise EDA — Findings

The GLORYS temperature field exhibits a clear vertical thermal
structure across the analysis period.

Temperature and its variability change systematically with depth,
providing the reference subsurface structure used to train ANTARBODH.

Target-depth availability varies with depth, with deeper levels
generally having lower spatial coverage than shallow levels.

All ANTARBODH target depths are within or sufficiently supported by
the native GLORYS vertical range for the intended reconstruction.

The 1000 m target is bracketed by native GLORYS levels and can
therefore be obtained through vertical interpolation.

The 0 m target requires special handling because the shallowest native
GLORYS level is approximately 0.49 m. The shallowest native level will
be used as a proxy rather than applying blind vertical extrapolation.

The target-depth support checks in this section are diagnostic only.
Final target fields will be generated during the preprocessing stage
using the defined vertical interpolation strategy.

### Preprocessing implications

- retain the full native GLORYS depth structure during preprocessing
- vertically interpolate to the 15 ANTARBODH target depths
- use the shallowest native level as the 0 m proxy
- do not extrapolate beyond the supported GLORYS depth range
- preserve target masks
- do not globally fill missing target temperatures
- verify target-depth coverage after interpolation

## 7. Multi-variable Relationship EDA

The objective of this section is to investigate relationships between
surface observations and subsurface GLORYS temperature.

The analysis examines:

- SST vs subsurface temperature
- SSS vs subsurface temperature
- SSH/SLA vs subsurface temperature
- surface current U/V vs subsurface temperature
- surface wind U/V vs subsurface temperature
- relationships at multiple subsurface depths

This section is exploratory.

The correlation analysis is used to understand whether individual
surface variables exhibit linear spatial relationships with subsurface
temperature.

It is not used as a measure of model performance.

Because the source datasets have different native spatial resolutions
and observation footprints, the initial analysis is performed as a
representative one-day spatial diagnostic.

A full-period relationship analysis will be performed after temporal
and spatial harmonization to the common 0.25° grid.

## 7.1 Why Cross-Variable Relationship Analysis?

ANTARBODH is based on the hypothesis that surface ocean observations
contain information about the subsurface thermal structure.

Before training a machine-learning model, it is useful to investigate
whether measurable statistical relationships exist between the surface
inputs and subsurface temperature.

For example:

Surface SST
     ↓
may contain information about
     ↓
temperature at 50 m, 100 m, 200 m, etc.

However, SST alone is not expected to completely describe the subsurface
ocean.

Other variables such as salinity, sea-level anomaly, currents, and winds
may provide additional information.

Therefore, this section evaluates both individual relationships and
the combined set of surface observables.

Correlation is used as an exploratory measure, not as proof of causality.
A low correlation between an individual surface variable and a deep
temperature field does not imply that the variable is useless for a
nonlinear machine-learning model.

In [ ]:
# ============================================================
# Convert SST from Kelvin to Celsius
# ============================================================

sst_c = (
    sst["sea_surface_temperature"]
    - 273.15
)

sst_c.attrs = (
    sst["sea_surface_temperature"]
    .attrs.copy()
)

sst_c.attrs["units"] = "degrees_Celsius"

print("Type:", type(sst_c))
print("Dimensions:", sst_c.dims)
print("Units:", sst_c.attrs["units"])

print(
    "SST range:",
    float(sst_c.min().compute().item()),
    "to",
    float(sst_c.max().compute().item()),
    "°C"
)

In [ ]:
# Extract SST variable and convert Kelvin → Celsius


sst_c.attrs = sst["sea_surface_temperature"].attrs.copy()
sst_c.attrs["units"] = "degrees_Celsius"

print("Type:", type(sst_c))
print("Dimensions:", sst_c.dims)
print("Units:", sst_c.attrs["units"])
print(
    "SST range:",
    float(sst_c.min().compute().item()),
    "to",
    float(sst_c.max().compute().item()),
    "°C"
)

In [ ]:
# ============================================================
# Select Representative GLORYS Surface Temperature
# ============================================================

correlation_date = "2020-01-15"

glorys_surface = (
    glorys["thetao"]
    .sel(
        time=correlation_date,
        method="nearest"
    )
    .isel(depth=0)
)

print(
    "GLORYS type:",
    type(glorys_surface)
)

print(
    "GLORYS dimensions:",
    glorys_surface.dims
)

print(
    "GLORYS depth:",
    float(
        glorys["depth"]
        .isel(depth=0)
        .compute()
        .item()
    ),
    "m"
)

print(
    "GLORYS selected time:",
    glorys_surface["time"].values
)

In [ ]:
sst_surface = (
    sst_c
    .sel(
        time=correlation_date,
        method="nearest"
    )
)

print(
    "SST dimensions:",
    sst_surface.dims
)

print(
    "SST selected time:",
    sst_surface["time"].values
)



In [ ]:
# ============================================================
# Interpolate GLORYS Surface Temperature to SST Grid
# ============================================================

glorys_surface_on_sst = (
    glorys_surface
    .interp(
        latitude=sst_surface.latitude,
        longitude=sst_surface.longitude
    )
)

print(
    "Interpolated GLORYS type:",
    type(glorys_surface_on_sst)
)

print(
    "Dimensions:",
    glorys_surface_on_sst.dims
)

print(
    "Shape:",
    glorys_surface_on_sst.shape
)

In [ ]:
# ============================================================
# Valid SST–GLORYS Comparison Mask
# ============================================================

valid_surface = (
    np.isfinite(sst_surface)
    & np.isfinite(glorys_surface_on_sst)
)

print(
    "Mask type:",
    type(valid_surface)
)

print(
    "Mask shape:",
    valid_surface.shape
)

In [ ]:
# ============================================================
# Valid Paired Observations
# ============================================================

valid_count = (
    valid_surface
    .sum()
    .compute()
    .item()
)

valid_percentage = (
    valid_surface
    .mean()
    .compute()
    .item()
    * 100
)

print(
    "Valid SST–GLORYS surface comparison points:",
    f"{int(valid_count):,}"
)

print(
    "Valid comparison percentage:",
    f"{valid_percentage:.2f}%"
)

In [ ]:
# ============================================================
# SST vs GLORYS Surface Temperature Correlation
# ============================================================

surface_corr = xr.corr(
    sst_surface.where(valid_surface),
    glorys_surface_on_sst.where(valid_surface)
)

surface_corr_value = (
    surface_corr
    .compute()
    .item()
)

print(
    "SST vs GLORYS surface temperature correlation:",
    f"{surface_corr_value:.4f}"
)

In [ ]:
# ============================================================
# Prepare Controlled Scatter Sample
# ============================================================

sst_values = (
    sst_surface
    .where(valid_surface)
    .values
    .ravel()
)

glorys_values = (
    glorys_surface_on_sst
    .where(valid_surface)
    .values
    .ravel()
)

valid = (
    np.isfinite(sst_values)
    & np.isfinite(glorys_values)
)

sst_values = sst_values[valid]
glorys_values = glorys_values[valid]

print(
    "Available valid scatter points:",
    f"{len(sst_values):,}"
)

In [ ]:
# ============================================================
# Limit Scatter Plot Sample Size
# ============================================================

max_scatter_points = 10000

if len(sst_values) > max_scatter_points:

    rng = np.random.default_rng(42)

    sample_indices = rng.choice(
        len(sst_values),
        size=max_scatter_points,
        replace=False
    )

    sst_plot = sst_values[
        sample_indices
    ]

    glorys_plot = glorys_values[
        sample_indices
    ]

else:

    sst_plot = sst_values
    glorys_plot = glorys_values

print(
    "Points used in scatter plot:",
    f"{len(sst_plot):,}"
)

In [ ]:
# ============================================================
# SST vs GLORYS Surface Temperature
# ============================================================

plt.figure(figsize=(8, 7))

plt.scatter(
    sst_plot,
    glorys_plot,
    s=8,
    alpha=0.35
)

# ------------------------------------------------------------
# 1:1 reference line
# ------------------------------------------------------------

minimum = min(
    sst_plot.min(),
    glorys_plot.min()
)

maximum = max(
    sst_plot.max(),
    glorys_plot.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--",
    linewidth=1.5
)

plt.xlabel(
    "Satellite SST (°C)"
)

plt.ylabel(
    "GLORYS Surface Temperature (°C)"
)

plt.title(
    "SST vs GLORYS Surface Temperature\n"
    f"{correlation_date}"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "sst_vs_glorys_surface_temperature.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# SST–GLORYS Relationship Summary
# ============================================================

print("SST–GLORYS Surface Relationship")
print("---------------------------------")

print(
    "Analysis date:",
    correlation_date
)

print(
    "Valid paired observations:",
    f"{int(valid_count):,}"
)

print(
    "Valid coverage:",
    f"{valid_percentage:.2f}%"
)

print(
    "Pearson correlation:",
    f"{surface_corr_value:.4f}"
)

### SST–GLORYS Surface Relationship Interpretation

The representative-day comparison evaluates the spatial linear
relationship between satellite SST and the shallowest GLORYS
temperature level.

The SST and GLORYS fields are first placed on a common spatial grid
for comparison. Only locations where both datasets contain finite
observations are included.

A strong positive correlation indicates that spatial variations in
surface temperature are associated with spatial variations in the
near-surface GLORYS temperature field.

This analysis is exploratory and does not represent model prediction
performance.

For the multi-year dataset, the same diagnostic can be repeated for
selected representative dates or seasons. A full-period relationship
analysis should be performed only after all variables are harmonized to
the common 0.25° × 0.25° grid.

The scatter plot uses a controlled sample of valid observations to
avoid excessive memory use while preserving the spatial relationship.

## 7.8 SST–Subsurface Temperature Relationship

The relationship between SST and GLORYS temperature is evaluated at
multiple depths.

The selected depths provide a broad representation of:

- near-surface water
- upper-ocean structure
- thermocline-region variability
- intermediate/deeper ocean
- deep ocean

Correlation is calculated only over spatial locations where both SST
and GLORYS temperature are valid.

In [ ]:
# ============================================================
# Representative Dates for Multi-season EDA
# ============================================================

representative_dates = [
    "2020-01-15",
    "2020-04-15",
    "2020-07-15",
    "2020-10-15",
    "2021-01-15",
    "2021-04-15",
    "2021-07-15",
    "2021-10-15",
    "2022-01-15",
    "2022-04-15",
    "2022-07-15",
    "2022-10-15",
    "2023-01-15",
    "2023-04-15",
    "2023-07-15",
    "2023-10-15",
    "2024-01-15",
    "2024-04-15",
    "2024-07-15",
    "2024-10-15",
]

print(
    "Representative dates:"
)

for date in representative_dates:
    print(" -", date)

In [ ]:
# ============================================================
# SST–GLORYS Correlation Across Representative Dates
# ============================================================

seasonal_correlations = {}

for date in representative_dates:

    sst_day = (
        sst_c
        .sel(
            time=date,
            method="nearest"
        )
    )

    glorys_day = (
        glorys["thetao"]
        .sel(
            time=date,
            method="nearest"
        )
        .isel(depth=0)
    )

    glorys_on_sst = (
        glorys_day
        .interp(
            latitude=sst_day.latitude,
            longitude=sst_day.longitude
        )
    )

    valid = (
        np.isfinite(sst_day)
        & np.isfinite(glorys_on_sst)
    )

    corr = xr.corr(
        sst_day.where(valid),
        glorys_on_sst.where(valid)
    )

    valid_count_day = (
        valid
        .sum()
        .compute()
        .item()
    )

    corr_value = (
        corr
        .compute()
        .item()
    )

    seasonal_correlations[
        str(date)
    ] = {
        "correlation": corr_value,
        "valid_count": int(valid_count_day),
    }

    print(
        f"{date} → "
        f"r = {corr_value:.4f}, "
        f"n = {int(valid_count_day):,}"
    )

In [ ]:
# ============================================================
# Representative-date SST–GLORYS Correlations
# ============================================================

dates = list(
    seasonal_correlations.keys()
)

correlations = [
    seasonal_correlations[date][
        "correlation"
    ]
    for date in dates
]

plt.figure(figsize=(9, 5))

plt.bar(
    dates,
    correlations
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.ylabel(
    "Pearson Correlation (r)"
)

plt.xlabel(
    "Representative Date"
)

plt.title(
    "SST vs GLORYS Surface Temperature "
    "Across Representative Dates"
)

plt.ylim(-1, 1)

plt.xticks(
    rotation=30
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "sst_glorys_representative_date_correlation.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Test depths for SST–subsurface temperature relationship
test_depths = [
    5,
    50,
    100,
    200,
    500,
    700,
    1000
]

# Representative date for raw-data EDA
correlation_date = "2020-01-15"

# Select SST for the representative date
sst_surface = (
    sst_c
    .sel(
        time=correlation_date,
        method="nearest"
    )
)

print(
    "Correlation date:",
    str(sst_surface.time.values)
)

sst_depth_correlations = []

for target_depth in test_depths:

    # Select the representative GLORYS day
    theta_depth = (
        glorys["thetao"]
        .sel(
            time=correlation_date,
            method="nearest"
        )
        .interp(
            depth=target_depth
        )
    )

    # Temporarily interpolate GLORYS horizontally to SST grid
    theta_on_sst = theta_depth.interp(
        latitude=sst_surface.latitude,
        longitude=sst_surface.longitude
    )

    # Valid comparison mask
    valid = (
        np.isfinite(sst_surface)
        & np.isfinite(theta_on_sst)
    )

    # Pearson correlation
    corr = xr.corr(
        sst_surface.where(valid),
        theta_on_sst.where(valid)
    )

    corr_value = corr.compute().item()

    sst_depth_correlations.append(corr_value)

    valid_count = valid.sum().compute().item()

    print(
        f"{target_depth:>4} m : "
        f"correlation = {corr_value:.4f} | "
        f"valid points = {valid_count}"
    )


# Convert correlation results to xarray DataArray
sst_corr_profile = xr.DataArray(
    sst_depth_correlations,
    coords={
        "depth": test_depths
    },
    dims=["depth"],
    name="sst_temperature_correlation"
)

sst_corr_profile.attrs = {
    "description": (
        "Pearson correlation between satellite SST "
        "and GLORYS temperature at selected depths"
    ),
    "units": "correlation coefficient",
    "source": "Raw prototype data",
    "date": correlation_date,
    "spatial_interpolation": (
        "GLORYS horizontally interpolated to SST grid"
    ),
    "vertical_interpolation": (
        "GLORYS temperature vertically interpolated "
        "to selected depths"
    ),
    "note": (
        "Temporary spatial and vertical interpolation "
        "used for exploratory analysis only"
    )
}

print("\nSST–subsurface correlation profile:")
print(sst_corr_profile)


# Plot correlation profile
plt.figure(figsize=(8, 7))

plt.plot(
    sst_corr_profile.values,
    sst_corr_profile.depth.values,
    marker="o"
)

plt.gca().invert_yaxis()

plt.axvline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel(
    "Pearson Correlation: SST vs Temperature"
)

plt.ylabel("Depth (m)")

plt.title(
    f"SST–Subsurface Temperature Correlation\n"
    f"{correlation_date}"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "sst_subsurface_temperature_correlation_20200115.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


# Save correlation profile
sst_corr_profile.to_netcdf(
    OUTPUT_DIR / "sst_subsurface_temperature_correlation_20200115.nc"
)

In [ ]:
# ============================================================
# SST vs GLORYS Temperature Correlation
# Four Representative Dates
# ============================================================

test_depths = [
    5,
    50,
    100,
    200,
    500,
    700,
    1000
]

representative_dates = [
    "2020-01-15",
    "2020-04-15",
    "2020-07-15",
    "2020-10-15"
]

season_labels = {
    "2020-01-15": "Winter",
    "2020-04-15": "Pre-monsoon",
    "2020-07-15": "Monsoon",
    "2020-10-15": "Post-monsoon"
}

# Store results
all_correlations = []

# ------------------------------------------------------------
# Loop through representative dates
# ------------------------------------------------------------

for correlation_date in representative_dates:

    print("\n" + "=" * 60)

    print(
        f"Date: {correlation_date} "
        f"({season_labels[correlation_date]})"
    )

    print("=" * 60)

    # --------------------------------------------------------
    # Select SST
    # --------------------------------------------------------

    sst_surface = (
        sst_c
        .sel(
            time=correlation_date,
            method="nearest"
        )
    )

    selected_date = str(
        sst_surface.time.values
    )

    print(
        "Selected date:",
        selected_date
    )

    date_correlations = []

    # --------------------------------------------------------
    # Loop through depths
    # --------------------------------------------------------

    for target_depth in test_depths:

        # Select GLORYS for same date
        theta_depth = (
            glorys["thetao"]
            .sel(
                time=correlation_date,
                method="nearest"
            )
            .interp(
                depth=target_depth
            )
        )

        # Interpolate GLORYS horizontally
        # onto SST grid
        theta_on_sst = theta_depth.interp(
            latitude=sst_surface.latitude,
            longitude=sst_surface.longitude
        )

        # Valid comparison mask
        valid = (
            np.isfinite(sst_surface)
            & np.isfinite(theta_on_sst)
        )

        # Pearson correlation
        corr = xr.corr(
            sst_surface.where(valid),
            theta_on_sst.where(valid)
        )

        corr_value = corr.compute().item()

        valid_count = (
            valid.sum()
            .compute()
            .item()
        )

        date_correlations.append(
            corr_value
        )

        print(
            f"{target_depth:>4} m : "
            f"correlation = {corr_value:.4f} | "
            f"valid points = {valid_count}"
        )

    # --------------------------------------------------------
    # Store results for this date
    # --------------------------------------------------------

    all_correlations.append(
        date_correlations
    )


# ------------------------------------------------------------
# Create xarray DataArray
# ------------------------------------------------------------

sst_seasonal_corr = xr.DataArray(
    np.array(all_correlations),
    coords={
        "date": representative_dates,
        "depth": test_depths
    },
    dims=[
        "date",
        "depth"
    ],
    name="sst_temperature_correlation"
)

sst_seasonal_corr.attrs = {
    "description": (
        "Pearson correlation between satellite SST "
        "and GLORYS temperature at selected depths "
        "for representative dates"
    ),
    "units": "correlation coefficient",
    "source": "Raw prototype data",
    "spatial_interpolation": (
        "GLORYS horizontally interpolated to SST grid"
    ),
    "vertical_interpolation": (
        "GLORYS vertically interpolated to selected depths"
    ),
    "note": (
        "Temporary spatial and vertical interpolation "
        "used for exploratory analysis only"
    )
}

print("\n")
print("=" * 60)
print("SST–Subsurface Correlation Profile")
print("=" * 60)

print(sst_seasonal_corr)

In [ ]:
# ============================================================
# Plot SST–Subsurface Correlation Profiles
# ============================================================

plt.figure(figsize=(9, 7))

for date in representative_dates:

    plt.plot(
        sst_seasonal_corr.sel(
            date=date
        ).values,
        sst_seasonal_corr.depth.values,
        marker="o",
        label=(
            f"{season_labels[date]} "
            f"({date})"
        )
    )

plt.gca().invert_yaxis()

plt.axvline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel(
    "Pearson Correlation: SST vs Temperature"
)

plt.ylabel("Depth (m)")

plt.title(
    "SST–Subsurface Temperature Correlation\n"
    "Representative Seasonal Dates"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR
    / "sst_subsurface_temperature_correlation_seasonal.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# Save Seasonal Correlation Results
# ============================================================

sst_seasonal_corr.to_netcdf(
    OUTPUT_DIR
    / "sst_subsurface_temperature_correlation_seasonal.nc"
)

print(
    "Saved:",
    OUTPUT_DIR
    / "sst_subsurface_temperature_correlation_seasonal.nc"
)

## Step 7 Inference:
 The one-day analysis shows a strong positive relationship between satellite SST and GLORYS near-surface temperature (r = 0.857). However, the correlation decreases sharply with depth, reaching approximately zero by 50 m for SST alone. This indicates that SST alone does not contain a sufficiently strong linear signal for reconstructing the deeper temperature profile. The result motivates AntarBodh's multivariate approach, in which SST, SSS, SSH/SLA, surface currents, and winds are jointly used to learn the nonlinear relationship between surface observations and subsurface temperature. These findings are preliminary because they are based on a single day and must be reassessed over the full temporal dataset.

In [30]:
correlation_date = "2020-01-15"

surface_variables = {
    "SST": sst_c.sel(time=correlation_date, method="nearest"),
    "SSH": ssh["sla"].sel(time=correlation_date, method="nearest"),
    "Current_U": currents["uo"].sel(time=correlation_date, method="nearest").squeeze(),
    "Current_V": currents["vo"].sel(time=correlation_date, method="nearest").squeeze(),
    "Wind_U": winds["eastward_wind"].sel(time=correlation_date, method="nearest"),
    "Wind_V": winds["northward_wind"].sel(time=correlation_date, method="nearest"),
}

In [31]:
sss_asc = sss_asc["Sea_Surface_Salinity"].sel(time=correlation_date, method="nearest") 
sss_desc = sss_desc["Sea_Surface_Salinity"].sel(time=correlation_date, method="nearest")


# Combine SSS Ascending and Descending into a single variable temporarily for EDA purposes


sss_combined = xr.where(
    sss_asc.notnull() & sss_desc.notnull(),
    (sss_asc + sss_desc) / 2,
    xr.where(
        sss_asc.notnull(),
        sss_asc,
        sss_desc
    )
)
#Both Valid-Average, if only one valid, take that value, if both invalid, result is NaN
surface_variables["SSS"] = sss_combined

In [ ]:
for name, variable in surface_variables.items():

    print(
        f"{name:12s} | "
        f"shape={variable.shape} | "
        f"min={float(variable.min().compute().item()):.3f} | "
        f"max={float(variable.max().compute().item()):.3f}"
    )

In [33]:
relationship_depths = [
    5,
    50,
    100,
    200,
    500,
    700,
    1000
]

In [34]:
representative_dates = [
    "2020-01-15",  # Winter
    "2020-04-15",  # Pre-monsoon
    "2020-07-15",  # Monsoon
    "2020-10-15",  # Post-monsoon
]

season_labels = {
    "2020-01-15": "Winter",
    "2020-04-15": "Pre-monsoon",
    "2020-07-15": "Monsoon",
    "2020-10-15": "Post-monsoon",
}

In [ ]:
surface_variables_by_date = {}

for date in representative_dates:

    sst_date = sst_c.sel(
        time=date,
        method="nearest"
    )

    ssh_date = ssh["sla"].sel(
        time=date,
        method="nearest"
    )

    current_u_date = currents["uo"].sel(
        time=date,
        method="nearest"
    ).squeeze()

    current_v_date = currents["vo"].sel(
        time=date,
        method="nearest"
    ).squeeze()

    wind_u_date = winds["eastward_wind"].sel(
        time=date,
        method="nearest"
    )

    wind_v_date = winds["northward_wind"].sel(
        time=date,
        method="nearest"
    )

    sss_asc_date = sss_asc["Sea_Surface_Salinity"].sel(
        time=date,
        method="nearest"
    )

    sss_desc_date = sss_desc["Sea_Surface_Salinity"].sel(
        time=date,
        method="nearest"
    )

    sss_combined_date = xr.where(
        sss_asc_date.notnull() & sss_desc_date.notnull(),
        (sss_asc_date + sss_desc_date) / 2,
        xr.where(
            sss_asc_date.notnull(),
            sss_asc_date,
            sss_desc_date
        )
    )

    surface_variables_by_date[date] = {
        "SST": sst_date,
        "SSS": sss_combined_date,
        "SSH": ssh_date,
        "Current_U": current_u_date,
        "Current_V": current_v_date,
        "Wind_U": wind_u_date,
        "Wind_V": wind_v_date,
    }

In [38]:
glorys_by_date = {}

for date in representative_dates:

    glorys_by_date[date] = (
        glorys["thetao"]
        .sel(
            time=date,
            method="nearest"
        )
    )

In [ ]:
correlation_results = {}

for date in representative_dates:

    print(
        f"Preparing correlations for "
        f"{date} ({season_labels[date]})"
    )

    theta_selected = glorys_by_date[date]

    date_results = {}

    for variable_name, surface_variable in (
        surface_variables_by_date[date].items()
    ):

        # Interpolate surface variable to GLORYS grid
        surface_on_glorys = surface_variable.interp(
            latitude=glorys.latitude,
            longitude=glorys.longitude
        )

        variable_correlations = []

        for target_depth in relationship_depths:

            # Vertical interpolation
            theta_depth = theta_selected.interp(
                depth=target_depth
            )

            # Valid comparison mask
            valid = (
                np.isfinite(surface_on_glorys)
                & np.isfinite(theta_depth)
            )

            # Keep lazy — do not compute here
            corr = xr.corr(
                surface_on_glorys.where(valid),
                theta_depth.where(valid)
            )

            variable_correlations.append(corr)

        date_results[variable_name] = variable_correlations

    correlation_results[date] = date_results

In [40]:
correlation_lazy = xr.DataArray(
    [
        [
            [
                correlation_results[date][variable][depth_index]
                for depth_index in range(
                    len(relationship_depths)
                )
            ]
            for variable in correlation_results[date]
        ]
        for date in representative_dates
    ],
    coords={
        "date": representative_dates,
        "surface_variable": [
            "SST",
            "SSS",
            "SSH",
            "Current_U",
            "Current_V",
            "Wind_U",
            "Wind_V"
        ],
        "depth": relationship_depths
    },
    dims=[
        "date",
        "surface_variable",
        "depth"
    ],
    name="surface_subsurface_temperature_correlation"
)

In [ ]:
correlation_all_dates = correlation_lazy.compute()
print(correlation_all_dates)

In [42]:
correlation_all_dates.attrs = {
    "description": (
        "Pearson correlation between surface observations "
        "and GLORYS temperature at selected depths "
        "for representative dates"
    ),
    "units": "correlation coefficient",
    "source": "Raw prototype data",
    "spatial_interpolation": (
        "Surface variables interpolated to GLORYS horizontal grid"
    ),
    "vertical_interpolation": (
        "GLORYS temperature vertically interpolated "
        "to selected depths"
    ),
    "note": (
        "Temporary spatial and vertical interpolation used "
        "for exploratory analysis only"
    ),
}

In [ ]:

#Get an individual correlation matrix for a specific date (e.g., winter)

winter_correlation_matrix = correlation_all_dates.sel(
    date="2020-01-15"
)

print(winter_correlation_matrix)

In [ ]:
mean_correlation_matrix = (
    correlation_all_dates
    .mean(dim="date")
)

print(mean_correlation_matrix)

In [ ]:
absolute_correlation_all_dates = abs(correlation_all_dates)


mean_absolute_correlation = (
    absolute_correlation_all_dates
    .mean(dim="date")
)

print(mean_absolute_correlation)

In [ ]:
for date in representative_dates:

    matrix = correlation_all_dates.sel(
        date=date
    )

    plt.figure(figsize=(10, 6))

    plt.imshow(
        matrix.values,
        aspect="auto",
        interpolation="nearest",
        vmin=-1,
        vmax=1
    )

    plt.colorbar(
        label="Pearson Correlation"
    )

    plt.xticks(
        range(len(relationship_depths)),
        relationship_depths
    )

    plt.yticks(
        range(len(matrix.surface_variable)),
        matrix.surface_variable.values
    )

    plt.xlabel("Depth (m)")
    plt.ylabel("Surface Variable")

    plt.title(
        f"Surface Variables vs GLORYS Subsurface Temperature\n"
        f"{season_labels[date]} ({date})"
    )

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR
        / f"surface_variables_vs_subsurface_temperature_{date}.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.imshow(
    mean_correlation_matrix.values,
    aspect="auto",
    interpolation="nearest",
    vmin=-1,
    vmax=1
)

plt.colorbar(
    label="Mean Pearson Correlation"
)

plt.xticks(
    range(len(relationship_depths)),
    relationship_depths
)

plt.yticks(
    range(len(mean_correlation_matrix.surface_variable)),
    mean_correlation_matrix.surface_variable.values
)

plt.xlabel("Depth (m)")
plt.ylabel("Surface Variable")

plt.title(
    "Mean Surface Variables vs GLORYS Subsurface Temperature\n"
    "Four Representative Dates"
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR
    / "surface_variables_vs_subsurface_temperature_mean.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [48]:
correlation_all_dates.to_netcdf(
    OUTPUT_DIR
    / "surface_subsurface_temperature_correlations_4dates.nc"
)

In [49]:
mean_correlation_matrix.to_netcdf(
    OUTPUT_DIR
    / "surface_subsurface_temperature_correlations_mean.nc"
)

## 7.19 Cross-Variable EDA Interpretation

The cross-variable analysis will be interpreted using the following
principles:

### SST

SST is expected to have the strongest relationship with temperature
near the surface. Its relationship may weaken with increasing depth.

If SST retains meaningful correlation at subsurface depths, this provides
evidence that surface thermal conditions contain information about the
subsurface thermal structure.

### SSS

SSS may provide complementary information because salinity contributes
to seawater density and ocean stratification.

However, the SSS observations are substantially sparser than SST, so
correlation values must be interpreted together with observation
coverage.

### SSH/SLA

SSH/SLA may contain information associated with large-scale ocean
dynamics, eddies, and subsurface structure.

Its relationship with temperature may therefore remain meaningful at
depths where SST alone becomes less informative.

### Surface Currents

Current U and V components represent horizontal ocean dynamics.

Their individual linear correlation with temperature may be modest,
but they may still provide useful nonlinear information to a machine
learning model.

### Surface Winds

Surface winds represent atmospheric forcing and can influence mixing
and upper-ocean structure.

Their direct linear correlation with temperature at depth may be weak,
but they may provide complementary predictive information.

### Important limitation

Correlation measures only linear statistical association.

A low Pearson correlation does not mean that a variable contains no
useful information for ANTARBODH. A neural network can learn nonlinear
and multivariate relationships that are not captured by pairwise
correlation.

Therefore, this analysis is used for EDA and feature understanding,
not for automatically removing variables.

### Feature Selection Policy

The correlation analysis will not be used to automatically remove
surface variables.

All seven planned input channels will initially be retained:

SST, SSS, SSH, Current U, Current V, Wind U, Wind V.

Feature removal will only be considered later if there is a strong
data-quality, redundancy, or experimental reason.

The purpose of this EDA is to understand the data and guide modelling
experiments rather than to prematurely simplify the input feature set.

## Step 7 — Multi-Variable Relationship Analysis: Inference

The one-day prototype was used to investigate the spatial Pearson correlation
between seven surface observations and GLORYS subsurface temperature at selected
depths.

The results show that no single surface variable exhibits a strong relationship
with subsurface temperature across the full depth range.

SST shows the strongest relationship at 5 m (r = 0.857), but its correlation
decreases rapidly with depth. In contrast, SSH exhibits substantially stronger
relationships in the upper subsurface, reaching r = 0.721 at 100 m and r = 0.587
at 200 m. SSS shows a moderate near-surface relationship (r = 0.479 at 5 m),
while surface current and wind components exhibit weaker but depth-dependent
relationships.

These results provide preliminary evidence that different surface observations
contain complementary information about different portions of the subsurface
thermal structure.

This supports the use of a multivariate input representation containing SST,
SSS, SSH, surface currents, and surface winds rather than relying on SST alone.

However, these Pearson correlations represent only linear spatial associations
for a single day and should not be interpreted as causal relationships or as
direct measures of model predictive performance. Weak individual correlations
also do not imply that a variable contains no predictive information, because
the final model may learn nonlinear interactions between multiple surface
variables.

The results will therefore be treated as exploratory evidence supporting the
multivariate AntarBodh architecture. The actual reconstruction capability will
be evaluated later using held-out GLORYS data and independent ARGO/INCOIS
observations.

# STEP 8 — Quality-Control Decisions

Before transforming the raw oceanographic datasets, we define quality-control
(QC) rules for each surface observation and the GLORYS target.

The purpose of QC is to distinguish between:

1. valid observations that should be retained,
2. physically implausible or invalid observations that should be masked,
3. missing observations that may be conservatively estimated, and
4. large unsupported gaps that should remain missing.

QC decisions are based on the statistical and spatial exploratory analysis
performed earlier in this notebook.

The QC process is intentionally conservative. Observations are not removed
solely because they are statistically extreme. Spatial structure, physical
meaning, missing-data patterns, dataset metadata, and variable-specific
characteristics are considered before defining a QC rule.

The same QC philosophy will later be applied consistently when the prototype
is scaled from one day to the full 2020–2025 period.

The output of this section is a documented set of QC rules that will be
implemented in the subsequent preprocessing steps.

```mermaid
flowchart TD
    A[RAW OBSERVATION] --> B{Is it missing?}
    B -->|YES| C[Leave missing for later treatment]
    B -->|NO| D{Is it valid?}
    D -->|YES| E[KEEP]
    D -->|NO| F[MASK]

```

Principle 1 — Don't delete observations just because they're extreme
Principle 2 — Use metadata-defined invalid/fill values first
Principle 3 — Physical plausibility checks come after metadata checks

### SST

Your prototype showed:


```bash
Missing: ~58%
Raw units: Kelvin
Range: ~296.94–302.53 K
```


Converted:

~23.79–29.38 °C

Those temperatures are plausible for the Bay of Bengal prototype.

* Decision

Keep all valid SST observations.

* Do:

Kelvin
  ↓
Celsius

* and mask only:

* explicit fill values
* NaN
* clearly invalid metadata-defined values

Do not remove the high end simply because it is high.

Missingness

SST has substantial missingness.

We should not globally fill it.

Later we can perform conservative spatial interpolation where appropriate and preserve a mask.

### SSS

This is our most delicate variable.

You found:

Ascending
```bash
valid: ~11.9%
missing: ~88.1%
range: 26.16–40.67
Descending
valid: ~32.6%
missing: ~67.4%
range: 25.77–35.86

```

And importantly:

* ascending and descending overlap was extremely small
* high ascending values did not have corresponding descending observations
* high ascending values showed coherent swath-like structure

Therefore we cannot justify simply deleting SSS >35 based on this prototype.

Decision

First:

SSS ascending
      +
SSS descending
      ↓
combined SSS

using:

both valid
    → mean

ascending only
    → ascending

descending only
    → descending

neither
    → NaN

But importantly, this combination should happen after basic invalid/fill-value handling.

Missingness

SSS is extremely sparse.

Therefore:

We must not aggressively spatially interpolate SSS across large gaps.

### SSH / SLA

Your SSH:


```bash
missing: ~15.9%
range: -0.184 to +0.255 m
```

This looks like a reasonable SLA-type field.

Decision

Keep valid observations.

Mask:

NaN
explicit fill/invalid values

Do not remove positive or negative values simply because they're relatively large.

The positive and negative values are part of the spatial SSH anomaly field.

Later:

conservative spatial interpolation may be applied to small gaps.

Large gaps remain masked.

### Surface currents

You have:

U
```bash
range ≈ -0.944 to +0.700 m/s
V
range ≈ -0.551 to +0.685 m/s
```

These are components of velocity.

This is important:

Current_U < 0

does not mean invalid.

It means the flow has a westward component.

Likewise:

Current_V < 0

means a southward component.

Decision

Do not filter negative currents.

Keep valid U/V values.

Mask only:

explicit fill values
NaN
clearly invalid metadata-defined values

We will preserve U and V as separate channels.

### Winds

Your wind fields showed:

missing: ~72.4%

with:

Wind_U: approximately -10 to +2 m/s
Wind_V: approximately -8 to +2.5 m/s

The high missingness is consistent with the swath-like observational coverage we saw.

Very important:

Do not interpret:

Wind_U = -8 m/s

as an invalid value.

It represents wind direction.

Decision

Keep valid wind observations.

Mask:

explicit invalid/fill values
NaN

Do not fill the entire domain with zero.

And because the swath coverage is sparse:

Large gaps should remain masked rather than being aggressively interpolated.

### GLORYS target

GLORYS is different.

It's not a surface observation we're trying to clean for input.

It is our training/reference target.

We need to be particularly careful not to manufacture target values.

You found:

missingness increases with depth

approximately:

surface       ~19%
100 m         ~29–30%
500 m         ~34%
1000 m        ~37%

This is important.

Decision

Do not globally fill missing GLORYS temperature.

Instead:

valid GLORYS
     ↓
retain

invalid/missing GLORYS
     ↓
mask

Later, when constructing training samples, the target mask determines whether a location/depth can contribute to the loss.

| Variable | QC decision | Missing-data treatment |
|----------|-------------|------------------------|
| SST | Remove explicit invalid/fill values; convert K → °C | Preserve mask; conservative interpolation later |
| SSS ascending | Remove explicit invalid/fill values; investigate extremes | Preserve sparse coverage |
| SSS descending | Remove explicit invalid/fill values; investigate extremes | Preserve sparse coverage |
| Combined SSS | Combine valid ascending/descending observations | No aggressive gap filling |
| SSH/SLA | Remove explicit invalid/fill values | Conservative treatment of small gaps |
| Current U | Remove explicit invalid/fill values | Preserve mask; conservative treatment |
| Current V | Remove explicit invalid/fill values | Preserve mask; conservative treatment |
| Wind U | Remove explicit invalid/fill values | Preserve swath gaps |
| Wind V | Remove explicit invalid/fill values | Preserve swath gaps |
| GLORYS thetao | Remove explicit invalid/fill values | Do not globally fill target |

                 RAW DATA
                     │
                     ▼
          Metadata / fill-value QC
                     │
                     ▼
            Physical sanity checks
                     │
                     ▼
            Observation masks
                     │
                     ▼
       ┌─────────────┴─────────────┐
       │                           │
   Small gaps                 Large gaps
       │                           │
       ▼                           ▼
Conservative treatment       Remain masked
       │
       └─────────────┬─────────────┘
                     ▼
              Common 0.25° grid
                     │
                     ▼
             Model-ready X

                 5m      50m     100m    200m    500m    700m    1000m
SST             0.757    0.237    0.182   -0.010   0.007   -0.023   -0.148
SSS            -0.044    0.010    0.326    0.242   0.222    0.122   -0.028
SSH             0.134    0.604    0.719    0.528   0.126    0.092    0.057
Current_U      -0.290   -0.120   -0.019    0.027   0.033    0.023    0.041
Current_V       0.054    0.057    0.063    0.093   0.130    0.136    0.053
Wind_U         -0.585   -0.224   -0.072   -0.007  -0.217   -0.201   -0.228
Wind_V         -0.351   -0.185    0.058    0.074  -0.043   -0.062   -0.019

# STEP 9 — Basic Quality Control and Observation Masks

This section implements the QC decisions defined in Step 8.

The initial QC stage is intentionally conservative. It removes or masks only
values that are explicitly identified as invalid through dataset metadata or
are clearly unusable.

Valid observations are retained, including statistically extreme observations
that may represent real oceanographic variability.

For each variable, an explicit observation mask is created:

- 1 / True  → usable observation
- 0 / False → missing or invalid observation

Missing observations are not filled during this stage.

The observation masks will be retained throughout preprocessing and will later
be used to distinguish original observations from interpolated or reconstructed
values.

This separation is important for preventing artificial information from being
introduced into the model inputs and for maintaining traceability of the
observational coverage.

In [ ]:
# ---------------------------------------------------------
# 9.1 Inspect QC-related metadata
# ---------------------------------------------------------

qc_variables = {
    "SST": (sst, "sea_surface_temperature"),
    "SSS_ascending": (sss_asc, "Sea_Surface_Salinity"),
    "SSS_descending": (sss_desc, "Sea_Surface_Salinity"),
    "SSH": (ssh, "sla"),
    "Current_U": (currents, "uo"),
    "Current_V": (currents, "vo"),
    "Wind_U": (winds, "eastward_wind"),
    "Wind_V": (winds, "northward_wind"),
    "GLORYS": (glorys, "thetao"),
}

for name, (dataset, variable_name) in qc_variables.items():
    if variable_name in dataset:
        var = dataset[variable_name]

        print("\n" + "=" * 60)
        print(name)
        print("=" * 60)

        print("Variable:", variable_name)
        print("dtype:", var.dtype)

        print("Attributes:")
        for key, value in var.attrs.items():
            print(f"  {key}: {value}")

        print("Encoding:")
        for key, value in var.encoding.items():
            if key in [
                "_FillValue",
                "missing_value",
                "dtype",
                "scale_factor",
                "add_offset"
            ]:
                print(f"  {key}: {value}")
    else:
        print(f"\n{name}: Variable '{variable_name}' not found in the dataset.")

In [ ]:
print(type(sss_asc))
print(type(sss_desc))

In [ ]:
sst_raw = sst["sea_surface_temperature"]

sst_qc = sst_raw.where(np.isfinite(sst_raw))

sst_mask = sst_qc.notnull()

sst_valid_count = sst_mask.sum().compute().item()
sst_total_count = sst_mask.size

print("SST valid:", sst_valid_count)
print(
    "SST missing:",
    sst_total_count - sst_valid_count
)
print(
    "SST missing %:",
    (sst_total_count - sst_valid_count)
    / sst_total_count * 100
)

In [ ]:
sss_asc_raw = sss_asc["Sea_Surface_Salinity"]
sss_desc_raw = sss_desc["Sea_Surface_Salinity"]

# Basic finite-value QC
sss_asc_qc = sss_asc_raw.where(np.isfinite(sss_asc_raw))
sss_desc_qc = sss_desc_raw.where(np.isfinite(sss_desc_raw))

# Physical/anomaly QC
sss_asc_qc = sss_asc_qc.where(
    (sss_asc_qc >= 0) & (sss_asc_qc <= 38)
)

sss_desc_qc = sss_desc_qc.where(
    (sss_desc_qc >= 0) & (sss_desc_qc <= 38)
)
sss_asc_mask = sss_asc_qc.notnull()
sss_desc_mask = sss_desc_qc.notnull()

sss_asc_valid_count = (
    sss_asc_mask.sum()
    .compute()
    .item()
)

sss_desc_valid_count = (
    sss_desc_mask.sum()
    .compute()
    .item()
)

print(
    "SSS ascending valid:",
    sss_asc_valid_count
)

print(
    "SSS descending valid:",
    sss_desc_valid_count
)

In [ ]:
print("SSS ascending:")
print(
    "  min:",
    sss_asc_qc.min(skipna=True).compute().item()
)
print(
    "  max:",
    sss_asc_qc.max(skipna=True).compute().item()
)

print("\nSSS descending:")
print(
    "  min:",
    sss_desc_qc.min(skipna=True).compute().item()
)
print(
    "  max:",
    sss_desc_qc.max(skipna=True).compute().item()
)

In [ ]:
ssh_raw = ssh["sla"]

ssh_qc = ssh_raw.where(np.isfinite(ssh_raw))
ssh_mask = ssh_qc.notnull()

ssh_valid_count = (
    ssh_mask.sum()
    .compute()
    .item()
)

print(
    "SSH valid:",
    ssh_valid_count
)

In [ ]:
current_u_raw = currents["uo"]
current_v_raw = currents["vo"]

current_u_qc = current_u_raw.where(np.isfinite(current_u_raw))
current_v_qc = current_v_raw.where(np.isfinite(current_v_raw))

current_u_mask = current_u_qc.notnull()
current_v_mask = current_v_qc.notnull()

current_u_valid_count = (
    current_u_mask.sum()
    .compute()
    .item()
)

current_v_valid_count = (
    current_v_mask.sum()
    .compute()
    .item()
)

print("Current U valid:", current_u_valid_count)
print("Current V valid:", current_v_valid_count)

In [ ]:
wind_u_raw = winds["eastward_wind"]
wind_v_raw = winds["northward_wind"]

wind_u_qc = wind_u_raw.where(np.isfinite(wind_u_raw))
wind_v_qc = wind_v_raw.where(np.isfinite(wind_v_raw))

wind_u_mask = wind_u_qc.notnull()
wind_v_mask = wind_v_qc.notnull()

wind_u_valid_count = (
    wind_u_mask.sum()
    .compute()
    .item()
)

wind_v_valid_count = (
    wind_v_mask.sum()
    .compute()
    .item()
)

print("Wind U valid:", wind_u_valid_count)
print("Wind V valid:", wind_v_valid_count)

In [ ]:
thetao_raw = glorys["thetao"]

thetao_qc = thetao_raw.where(np.isfinite(thetao_raw))

thetao_mask = thetao_qc.notnull()

thetao_valid_count = (
    thetao_mask.sum()
    .compute()
    .item()
)

print(
    "GLORYS valid:",
    thetao_valid_count
)

In [ ]:
qc_report = {}

def add_qc_statistics(name, raw, qc):

    total_count = raw.size

    valid_count = (
        qc.notnull()
        .sum()
        .compute()
        .item()
    )

    rejected_count = total_count - valid_count

    qc_report[name] = {
        "total": total_count,
        "valid": valid_count,
        "rejected": rejected_count,
        "rejection_pct": rejected_count / total_count * 100,
        "retention_pct": valid_count / total_count * 100,
    }


add_qc_statistics("SST", sst_raw, sst_qc)
add_qc_statistics("SSS Ascending", sss_asc_raw, sss_asc_qc)
add_qc_statistics("SSS Descending", sss_desc_raw, sss_desc_qc)
add_qc_statistics("SSH", ssh_raw, ssh_qc)
add_qc_statistics("Current U", current_u_raw, current_u_qc)
add_qc_statistics("Current V", current_v_raw, current_v_qc)
add_qc_statistics("Wind U", wind_u_raw, wind_u_qc)
add_qc_statistics("Wind V", wind_v_raw, wind_v_qc)
add_qc_statistics("GLORYS thetao", thetao_raw, thetao_qc)


for name, stats in qc_report.items():
    print(
        f"{name:20s} | "
        f"Total: {stats['total']:,} | "
        f"Valid: {stats['valid']:,} | "
        f"Rejected: {stats['rejected']:,} | "
        f"Reject: {stats['rejection_pct']:.2f}% | "
        f"Retain: {stats['retention_pct']:.2f}%"
    )

In [ ]:
# ============================================================
# EXPORT QC REPORT
# ============================================================

import csv
from pathlib import Path

qc_report_path = Path("../outputs/qc_report.csv")

# Create output directory if it does not exist
qc_report_path.parent.mkdir(parents=True, exist_ok=True)

with open(qc_report_path, "w", newline="") as f:

    writer = csv.writer(f)

    # Header
    writer.writerow([
        "Dataset",
        "Total Observations",
        "Valid After QC",
        "Rejected",
        "Rejection %",
        "Retention %"
    ])

    # Data
    for name, stats in qc_report.items():
        writer.writerow([
            name,
            stats["total"],
            stats["valid"],
            stats["rejected"],
            round(stats["rejection_pct"], 4),
            round(stats["retention_pct"], 4)
        ])

print(f"QC report saved to: {qc_report_path}")

## Step 10:Physical Sanity Checks

In [ ]:
sst_c = sst_qc - 273.15

sst_physical = sst_c.where(
    (sst_c >= -2) & (sst_c <= 40)
)

print("SST °C:")
print(
    "min:",
    sst_physical.min().compute().item()
)

print(
    "max:",
    sst_physical.max().compute().item()
)

In [ ]:
sst_physical_bad = (
    (sst_c < -2)
    | (sst_c > 40)
)

sst_bad_count = (
    sst_physical_bad.sum()
    .compute()
    .item()
)

print(
    "SST outside physical sanity range:",
    sst_bad_count
)

print(
    "SST physical-range rejection %:",
    sst_bad_count / sst_c.size * 100
)

In [ ]:
print("SSS ascending:")
print(
    "min:",
    sss_asc_qc.min().compute().item()
)
print(
    "max:",
    sss_asc_qc.max().compute().item()
)

print("\nSSS descending:")
print(
    "min:",
    sss_desc_qc.min().compute().item()
)
print(
    "max:",
    sss_desc_qc.max().compute().item()
)

In [ ]:
ssh_physical = ssh_qc.where(
    (ssh_qc >= -5) & (ssh_qc <= 5)
)

print(
    "SSH min:",
    ssh_physical.min().compute().item()
)

print(
    "SSH max:",
    ssh_physical.max().compute().item()
)

In [ ]:
current_u_physical = current_u_qc.where(
    (current_u_qc >= -5) & (current_u_qc <= 5)
)

current_v_physical = current_v_qc.where(
    (current_v_qc >= -5) & (current_v_qc <= 5)
)

print(
    "Current U:",
    current_u_physical.min().compute().item(),
    current_u_physical.max().compute().item()
)

print(
    "Current V:",
    current_v_physical.min().compute().item(),
    current_v_physical.max().compute().item()
)

In [ ]:
wind_u_physical = wind_u_qc.where(
    (wind_u_qc >= -50) & (wind_u_qc <= 50)
)

wind_v_physical = wind_v_qc.where(
    (wind_v_qc >= -50) & (wind_v_qc <= 50)
)

print(
    "Wind U:",
    wind_u_physical.min().compute().item(),
    wind_u_physical.max().compute().item()
)

print(
    "Wind V:",
    wind_v_physical.min().compute().item(),
    wind_v_physical.max().compute().item()
)

In [ ]:
thetao_physical = thetao_qc.where(
    (thetao_qc >= -2) & (thetao_qc <= 40)
)

print("GLORYS:")
print(
    "min:",
    thetao_physical.min().compute().item()
)

print(
    "max:",
    thetao_physical.max().compute().item()
)

##  Step 11 — Unit standardization

* SST       → °C
* SSS       → salinity units as supplied
* SSH       → m
* Current U → m/s
* Current V → m/s
* Wind U    → m/s
* Wind V    → m/s
* GLORYS    → °C

In [ ]:
# ============================================================
# Step 11 — Unit Standardization
# ============================================================

# SST: Kelvin → Celsius
sst_c = sst_qc - 273.15
sst_c.attrs["units"] = "degrees_C"
sst_c.attrs["long_name"] = "Sea surface temperature"

sss_asc_std = sss_asc_qc
sss_desc_std = sss_desc_qc

ssh_std = ssh_qc

current_u_std = current_u_qc
current_v_std = current_v_qc

wind_u_std = wind_u_qc
wind_v_std = wind_v_qc

thetao_std = thetao_qc

print("Units after standardization:")
print("SST:", sst_c.attrs.get("units"))
print("SSS:", sss_asc_std.attrs.get("units"))
print("SSH:", ssh_std.attrs.get("units"))
print("Current U:", current_u_std.attrs.get("units"))
print("Current V:", current_v_std.attrs.get("units"))
print("Wind U:", wind_u_std.attrs.get("units"))
print("Wind V:", wind_v_std.attrs.get("units"))
print("GLORYS:", thetao_std.attrs.get("units"))

## Step 13 — Common 0.25° grid

In [ ]:
# ============================================================
# SSS — High-Value Anomaly Investigation
# ============================================================

print("SSS Ascending values > 50")
print("--------------------------------")

sss_asc_high = sss_asc_std.where(sss_asc_std > 50)

print("Count:",
      sss_asc_high.notnull().sum().compute().item())

print("Min:",
      sss_asc_high.min(skipna=True).compute().item())

print("Max:",
      sss_asc_high.max(skipna=True).compute().item())


print("\nSSS Descending values > 50")
print("--------------------------------")

sss_desc_high = sss_desc_std.where(sss_desc_std > 50)

print("Count:",
      sss_desc_high.notnull().sum().compute().item())

print("Min:",
      sss_desc_high.min(skipna=True).compute().item())

print("Max:",
      sss_desc_high.max(skipna=True).compute().item())

In [ ]:
sss_thresholds = [35, 36, 38, 40, 50, 55, 60]

print("========== SSS ASCENDING ==========")

for threshold in sss_thresholds:
    count = (
        (sss_asc_std > threshold)
        .sum()
        .compute()
        .item()
    )
    print(f"> {threshold}: {count:,}")


print("\n========== SSS DESCENDING ==========")

for threshold in sss_thresholds:
    count = (
        (sss_desc_std > threshold)
        .sum()
        .compute()
        .item()
    )
    print(f"> {threshold}: {count:,}")

In [ ]:
sss_asc_high_day = sss_asc_std.sel(
    time="2025-01-15",
    method="nearest"
).where(
    sss_asc_std.sel(
        time="2020-01-15",
        method="nearest"
    ) > 50
)

sss_asc_high_day.plot()

plt.title("SSS Ascending — Values > 50")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
sss_desc_high_day = sss_desc_std.sel(
    time="2020-01-15",
    method="nearest"
).where(
    sss_desc_std.sel(
        time="2020-01-15",
        method="nearest"
    ) > 50
)

sss_desc_high_day.plot()

plt.title("SSS Descending — Values > 50")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 13 — Define Common 0.25° Grid
# ============================================================

common_lat = xr.DataArray(
    np.arange(5.125, 20.0, 0.25),
    dims="latitude",
    name="latitude"
)

common_lon = xr.DataArray(
    np.arange(80.125, 100.0, 0.25),
    dims="longitude",
    name="longitude"
)

print("Common latitude:")
print(common_lat)

print("\nCommon longitude:")
print(common_lon)

print("\nGrid size:")
print("latitude:", common_lat.size)
print("longitude:", common_lon.size)

In [ ]:
sst_common = sst_c.interp(
    latitude=common_lat,
    longitude=common_lon
)

print(sst_common)

In [ ]:
# SSS ascending → common 0.25° grid
sss_asc_common = sss_asc_std.interp(
    latitude=common_lat,
    longitude=common_lon
)

# SSS descending → common 0.25° grid
sss_desc_common = sss_desc_std.interp(
    latitude=common_lat,
    longitude=common_lon
)

print("SSS ascending common grid:")
print(sss_asc_common)

print("\nSSS descending common grid:")
print(sss_desc_common)

In [ ]:
# ============================================================
# Combine SSS after regridding
# ============================================================

asc_valid = sss_asc_common.notnull()
desc_valid = sss_desc_common.notnull()

sss_common = xr.where(
    asc_valid & desc_valid,
    (sss_asc_common + sss_desc_common) / 2,
    xr.where(
        asc_valid,
        sss_asc_common,
        sss_desc_common
    )
)

sss_common.name = "SSS"

sss_common.attrs["long_name"] = (
    "Combined practical sea surface salinity"
)

sss_common.attrs["units"] = (
    sss_asc_qc.attrs.get("units")
)

print("\nCombined SSS:")
print(sss_common)

In [21]:
ssh_common = ssh_std.interp(
    latitude=common_lat,
    longitude=common_lon
)

In [22]:
current_u_common = current_u_std.interp(
    latitude=common_lat,
    longitude=common_lon
)

current_v_common = current_v_std.interp(
    latitude=common_lat,
    longitude=common_lon
)

In [23]:
wind_u_common = wind_u_std.interp(
    latitude=common_lat,
    longitude=common_lon
)

wind_v_common = wind_v_std.interp(
    latitude=common_lat,
    longitude=common_lon
)

In [29]:
current_u_common = current_u_common.sel(depth=0, drop=True)
current_v_common = current_v_common.sel(depth=0, drop=True) #drop extra depth layer from currents

In [ ]:
surface_common = xr.Dataset({
    "SST": sst_common,
    "SSS": sss_common,
    "SSH": ssh_common,
    "Current_U": current_u_common,
    "Current_V": current_v_common,
    "Wind_U": wind_u_common,
    "Wind_V": wind_v_common,
})

print(surface_common)

In [ ]:
# ============================================================
# Rechunk common surface dataset for downstream processing
# ============================================================

surface_common = surface_common.chunk({
    "time": 30,
    "latitude": 60,
    "longitude": 80,
})

print(surface_common.chunks)

In [ ]:
# ============================================================
# Common-grid coverage
# ============================================================

coverage = surface_common.notnull().mean(
    dim=("time", "latitude", "longitude")
)

coverage = coverage.compute()

for var in coverage.data_vars:
    valid_pct = coverage[var].item() * 100
    missing_pct = 100 - valid_pct

    print(
        f"{var:12s} | "
        f"valid={valid_pct:.2f}% | "
        f"missing={missing_pct:.2f}%"
    )

In [ ]:
import matplotlib.pyplot as plt

variables = [
    "SST",
    "SSS",
    "SSH",
    "Current_U",
    "Current_V",
    "Wind_U",
    "Wind_V",
]

for var in variables:
    plt.figure(figsize=(8, 5))

    surface_common[var].isel(time=0).plot()

    plt.title(f"{var} — Common 0.25° Grid")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR / f"{var.lower()}_common_grid_20200101.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

In [ ]:
for var in variables:
    plt.figure(figsize=(8, 5))

    surface_common[var].isel(time=0).notnull().plot()

    plt.title(f"{var} — Observation Mask")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

## Step 14 — Convert GLORYS to the 15 target depths

0, 5, 10, 20, 30, 50, 75, 100, 125, 150,
200, 300, 500, 700, 1000 m

In [ ]:
# ============================================================
# Step 14 — GLORYS Vertical Interpolation
# ============================================================

target_depths = xr.DataArray(
    [0, 5, 10, 20, 30, 50, 75, 100, 125, 150,
     200, 300, 500, 700, 1000],
    dims="depth",
    name="depth"
)

print(target_depths)

14.2 Handle the 0-m target carefully

GLORYS starts at approximately:

0.494 m

not exactly 0 m.

We should not extrapolate from the 0.494-m level to 0 m.

For this prototype, use the shallowest GLORYS level as the 0-m proxy:

In [ ]:
print("Original shallowest depth:",
      float(thetao_std.depth.values[0]))

print("Original deepest depth:",
      float(thetao_std.depth.values[-1])) #basically the shallowest one will be treated as 0m

In [ ]:

# ============================================================
# Prepare GLORYS depth coordinate for interpolation
# ============================================================

thetao_for_interp = thetao_std.assign_coords(
    depth=xr.where(
        thetao_std.depth == thetao_std.depth[0],
        0,
        thetao_std.depth
    )
)

print(thetao_for_interp.depth)

In [ ]:
print(
    "Interpolation depth range:",
    float(thetao_for_interp.depth.min().compute()),
    "→",
    float(thetao_for_interp.depth.max().compute())
)

print(
    "Target depth range:",
    float(target_depths.min()),
    "→",
    float(target_depths.max())
)

In [31]:
#vertical interpolation
thetao_15depth = thetao_for_interp.interp(
    depth=target_depths
)

In [ ]:
print("Target depths:")
print(thetao_15depth.depth.values)

In [ ]:
thetao_mean_profile = thetao_15depth.mean(
    dim=["time", "latitude", "longitude"],
    skipna=True
)

print(thetao_mean_profile)

In [ ]:
missing_profile = (
    100 * thetao_15depth.isnull().mean(
        dim=["time", "latitude", "longitude"]
    )
).compute()

for depth in target_depths:
    missing_pct = missing_profile.sel(depth=depth).item()

    print(
        f"{depth:4.0f} m | "
        f"missing = {missing_pct:.2f}%"
    )

## Step 15 — Regrid GLORYS horizontally

GLORYS native
181 × 241
      ↓
vertical interpolation
15 depths
      ↓
horizontal interpolation
60 × 80
      ↓
Y
(time, depth, latitude, longitude)

In [37]:
thetao_common = thetao_15depth.interp(
    latitude=common_lat,
    longitude=common_lon
)

In [ ]:
print(thetao_common)

In [ ]:
print("Shape:", thetao_common.shape)
print("Depths:", thetao_common.depth.values)
print("Lat:", thetao_common.latitude.values[[0, -1]])
print("Lon:", thetao_common.longitude.values[[0, -1]])

## Step 16 — Align everything and construct X/Y

X channels = 7
SST
SSS
SSH
Current_U
Current_V
Wind_U
Wind_V

Y depths = 15
0, 5, 10, 20, 30, 50, 75, 100, 125, 150,
200, 300, 500, 700, 1000 m

In [ ]:
# ============================================================
# Step 16 — Align Surface Inputs and GLORYS Target
# ============================================================

surface_common, thetao_common = xr.align(
    surface_common,
    thetao_common,
    join="exact"
)

print("Surface dimensions:")
print(surface_common.sizes)

print("\nTarget dimensions:")
print(thetao_common.sizes)

assert surface_common.sizes["time"] == thetao_common.sizes["time"]
assert surface_common.sizes["latitude"] == thetao_common.sizes["latitude"]
assert surface_common.sizes["longitude"] == thetao_common.sizes["longitude"]

In [ ]:
input_variables = [
    "SST",
    "SSS",
    "SSH",
    "Current_U",
    "Current_V",
    "Wind_U",
    "Wind_V",
]

X_da = xr.concat(
    [surface_common[var] for var in input_variables],
    dim="channel"
)

X_da = X_da.assign_coords(
    channel=input_variables
)

X_da.name = "surface_inputs"

print(X_da)

In [ ]:
Y_da = thetao_common.rename(
    "subsurface_temperature"
)

print(Y_da)

In [ ]:
X_mask = X_da.notnull()

X_mask.name = "surface_observation_mask"

print(X_mask)

In [51]:
Y_mask = Y_da.notnull()

Y_mask.name = "target_valid_mask"

In [ ]:
X_da = X_da.transpose(
    "time",
    "channel",
    "latitude",
    "longitude"
)

X_mask = X_mask.transpose(
    "time",
    "channel",
    "latitude",
    "longitude"
)

print("X shape:", X_da.shape)
print("Mask shape:", X_mask.shape)

In [ ]:
print("========== FINAL PROTOTYPE SHAPES ==========")

print("X:", X_da.shape)
print("Y:", Y_da.shape)
print("X mask:", X_mask.shape)
print("Y mask:", Y_mask.shape)

print("\nChannels:")
print(X_da.channel.values)

print("\nTarget depths:")
print(Y_da.depth.values)

In [ ]:
# ============================================================
# Rechunk input mask
# ============================================================

X_mask = X_mask.chunk({
    "time": 30,
    "channel": 7,
    "latitude": 60,
    "longitude": 80,
})

print(X_mask.chunks)

Step 17 — Define the missing-input strategy

observed value → keep it
missing value  → replace with a neutral value
mask           → tell model whether it was observed


raw X
 ↓
QC
 ↓
common grid
 ↓
split train/validation/test
 ↓
calculate training statistics
 ↓
normalize
 ↓
fill missing normalized values with 0
 ↓
provide observation mask separately

# Step 17 — Missing-Input Strategy

The surface observations have substantially different spatial coverage.
On the one-day prototype, only approximately 10.08% of grid cells contain
simultaneous observations from all seven surface variables.

Discarding all grid cells with any missing input would therefore remove most
of the available spatial domain.

ANTARBODH therefore preserves missing observations using an explicit
observation-mask representation.

For each surface variable:

- observed values are retained;
- missing values remain NaN during the preprocessing stage;
- a binary observation mask is retained;
- missing values will later be replaced by zero only after training-set
  normalization;
- the corresponding mask will be supplied to the model so that zero-imputed
  values are distinguishable from genuine zero-valued observations.

This allows the model to use partially observed surface fields without
treating missing observations as physical measurements.

No target temperature values are imputed at this stage.

In [ ]:
print(X_mask)

In [ ]:
channel_coverage = (
    100 * X_mask.mean(dim=["time", "latitude", "longitude"])
).compute()

print("========== INPUT COVERAGE ==========")

for channel in X_mask.channel.values:
    coverage = channel_coverage.sel(channel=channel).item()

    print(
        f"{channel:12s} | "
        f"coverage = {coverage:.2f}%"
    )

split time before normalization

This is where we need to start thinking about your eventual 5-year dataset.

Never calculate normalization statistics using the entire 5-year dataset before splitting.

That would leak information from validation/test periods into training.

Eventually:
```mermaid
2020–2023 → training
2024      → validation
2025      → test
```
or another chronological split you choose.

Then:
```mermaid
training data
      ↓
mean/std
      ↓
normalize train
normalize validation
normalize test
```


Validation and test use training statistics only.

For the current one-day prototype, we can't meaningfully create a realistic multi-year temporal split because we have only one day.

So don't calculate the final normalization statistics yet.

In [ ]:
target_coverage = (
    100 * Y_mask.mean(
        dim=["time", "latitude", "longitude"]
    )
).compute()

print("========== TARGET COVERAGE ==========")

for depth in Y_da.depth.values:
    coverage = target_coverage.sel(depth=depth).item()

    print(
        f"{depth:4.0f} m | "
        f"coverage = {coverage:.2f}%"
    )

In [ ]:
print(
    "Any valid target:",
    bool(Y_mask.any().compute())
)

## Step 29 — Finalize the prototype dataset

Before normalization, let's establish one important principle:

We do not need every target depth to be valid at every grid cell.

The target mask Y_mask will tell us which target values are available.

For training, later, the loss can be computed only over valid target locations.

So we preserve:

Y
+
Y_mask

rather than filling missing GLORYS temperatures.

In [ ]:
# ============================================================
# Step 29 — Create ML-ready Prototype Dataset
# ============================================================

processed = xr.Dataset(
    {
        "X": X_da,
        "X_mask": X_mask.astype(np.int8),
        "Y": Y_da,
        "Y_mask": Y_mask.astype(np.int8),
    }
)

processed.attrs["project"] = "ANTARBODH"
processed.attrs["description"] = (
    "ANTARBODH model-ready surface observations and "
    "GLORYS-derived subsurface temperature targets"
)

processed.attrs["time_start"] = str(
    processed.time.values[0]
)

processed.attrs["time_end"] = str(
    processed.time.values[-1]
)
processed.attrs["region"] = "Bay of Bengal prototype"
processed.attrs["grid_resolution"] = "0.25 degree"
processed.attrs["input_channels"] = 7
processed.attrs["target_depths"] = (
    "0, 5, 10, 20, 30, 50, 75, 100, 125, "
    "150, 200, 300, 500, 700, 1000 m"
)

print(processed)

In [ ]:
print("========== FINAL DIMENSION CHECK ==========")

print("X:")
print(processed["X"].dims, processed["X"].shape)

print("\nX mask:")
print(processed["X_mask"].dims, processed["X_mask"].shape)

print("\nY:")
print(processed["Y"].dims, processed["Y"].shape)

print("\nY mask:")
print(processed["Y_mask"].dims, processed["Y_mask"].shape)

In [ ]:
print("X NaNs:", int(processed["X"].isnull().sum()))
print("Y NaNs:", int(processed["Y"].isnull().sum()))

In [ ]:
for name in ["X", "Y"]:
    has_inf = bool(
        np.isinf(processed[name]).any().compute()
    )

    print(name, "has Inf:", has_inf)

In [ ]:
x_mask_correct = bool(
    (
        X_da.notnull()
        == processed["X_mask"].astype(bool)
    ).all().compute()
)

print("X mask correct:", x_mask_correct)

In [ ]:

y_mask_correct = bool(
    (
        Y_da.notnull()
        == processed["Y_mask"].astype(bool)
    ).all().compute()
)

print("Y mask correct:", y_mask_correct)

In [ ]:
print("Latitude:")
print(processed.latitude.values[[0, -1]])

print("\nLongitude:")
print(processed.longitude.values[[0, -1]])

print("\nDepth:")
print(processed.depth.values)

print("\nChannels:")
print(processed.channel.values)

In [ ]:
print("X time:")
print(X_da.time)

print("\nY time:")
print(Y_da.time)

print("\nNumber of X timesteps:", X_da.sizes["time"])
print("Number of Y timesteps:", Y_da.sizes["time"])

assert X_da.sizes["time"] == Y_da.sizes["time"]
assert X_da.time.equals(Y_da.time)

print("✓ X and Y time coordinates match")

In [ ]:
time_values = X_da.time.values

print(
    "Duplicate timestamps:",
    len(time_values) - len(np.unique(time_values))
)

In [ ]:
time_diff = X_da.time.diff("time")

print("Time differences:")
print(time_diff.to_pandas().value_counts())

In [ ]:
all_inputs_available = X_mask.all(dim="channel")

daily_input_coverage = (
    100 * all_inputs_available.mean(
        dim=["latitude", "longitude"]
    )
).compute()
daily_input_coverage

In [ ]:
daily_target_coverage = (
    100 * Y_mask.mean(
        dim=["depth", "latitude", "longitude"]
    )
).compute()
daily_target_coverage

In [ ]:
joint_target_input = (
    X_mask.any(dim="channel") &
    Y_mask.any(dim="depth")
)

overall_joint_fraction = (
    joint_target_input.mean().compute().item()
)

print(
    "Overall input-target spatial coverage:",
    overall_joint_fraction * 100,
    "%"
)

## Step 30 — Chronological split
```mermaid
TRAIN       VALIDATION       TEST

2020 ───────── 2023 | 2024 | 2025
       4 years       | 1 yr | 1 yr
```


In [ ]:
train_mask = X_da.time.dt.year <= 2023
val_mask = X_da.time.dt.year == 2024
test_mask = X_da.time.dt.year == 2025

In [ ]:
X_train = X_da.sel(time=train_mask)
X_val = X_da.sel(time=val_mask)
X_test = X_da.sel(time=test_mask)

Y_train = Y_da.sel(time=train_mask)
Y_val = Y_da.sel(time=val_mask)
Y_test = Y_da.sel(time=test_mask)

Xmask_train = X_mask.sel(time=train_mask)
Xmask_val = X_mask.sel(time=val_mask)
Xmask_test = X_mask.sel(time=test_mask)

Ymask_train = Y_mask.sel(time=train_mask)
Ymask_val = Y_mask.sel(time=val_mask)
Ymask_test = Y_mask.sel(time=test_mask)

In [ ]:
print("Train:", X_train.time.values[0], "→", X_train.time.values[-1])
print("Val:  ", X_val.time.values[0], "→", X_val.time.values[-1])
print("Test: ", X_test.time.values[0], "→", X_test.time.values[-1])

## step 31-Calculate normalization statistics from TRAIN ONLY


In [ ]:
train_mean = X_train.mean(
    dim=["time", "latitude", "longitude"],
    skipna=True
)

train_std = X_train.std(
    dim=["time", "latitude", "longitude"],
    skipna=True
)

In [ ]:
train_mean = train_mean.compute()
train_std = train_std.compute()

## step 32-Normalize X

In [ ]:
X_train_norm = (X_train - train_mean) / train_std
X_val_norm = (X_val - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std

## step 33-Handle missing input values

In [ ]:
X_train_norm = X_train_norm.fillna(0)
X_val_norm = X_val_norm.fillna(0)
X_test_norm = X_test_norm.fillna(0)

## step 34-Create the 14-channel model input

In [ ]:
def make_14_channel_input(X_norm, Xmask):
    mask_channels = Xmask.astype(np.float32)

    mask_channels = mask_channels.assign_coords(
        channel=[
            f"{c}_mask"
            for c in X_norm.channel.values
        ]
    )

    return xr.concat(
        [X_norm, mask_channels],
        dim="channel"
    )

In [ ]:
X_train_14 = make_14_channel_input(
    X_train_norm,
    Xmask_train
)

X_val_14 = make_14_channel_input(
    X_val_norm,
    Xmask_val
)

X_test_14 = make_14_channel_input(
    X_test_norm,
    Xmask_test
)

## step 35-Final dataset objects

In [ ]:
train_ds = xr.Dataset({
    "X": X_train_14,
    "Y": Y_train,
    "Y_mask": Ymask_train.astype(np.int8),
})

val_ds = xr.Dataset({
    "X": X_val_14,
    "Y": Y_val,
    "Y_mask": Ymask_val.astype(np.int8),
})

test_ds = xr.Dataset({
    "X": X_test_14,
    "Y": Y_test,
    "Y_mask": Ymask_test.astype(np.int8),
})

In [ ]:
print("========== FINAL ML SHAPES ==========")

print("Train X:", train_ds["X"].shape)
print("Train Y:", train_ds["Y"].shape)

print("Val X:", val_ds["X"].shape)
print("Val Y:", val_ds["Y"].shape)

print("Test X:", test_ds["X"].shape)
print("Test Y:", test_ds["Y"].shape)

In [ ]:
print(train_ds.X.channel.values)
print(train_ds.Y.depth.values)
assert train_ds.X.dims == (
    "time", "channel", "latitude", "longitude"
)

assert train_ds.Y.dims == (
    "time", "depth", "latitude", "longitude"
)

In [ ]:
train_norm_mean = X_train_norm.mean(
    dim=["time", "latitude", "longitude"],
    skipna=False
).compute()

train_norm_std = X_train_norm.std(
    dim=["time", "latitude", "longitude"],
    skipna=False
).compute()

print("Training normalized mean:")
print(train_norm_mean)

print("\nTraining normalized std:")
print(train_norm_std)

## step 36-Save only after everything passes